# 07G FORCE legacy geo stratigraphic fusion SINGLE NOTEBOOK GATE FIXED

> Saved outputs are from an earlier local run. Portable path edits have **not** been re-executed. See the repository README for setup and scientific limitations.


In [1]:
import sys
import subprocess
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scikit-learn": "sklearn",
    "joblib": "joblib",
    "pyarrow": "pyarrow",
}

for package_name, module_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package_name,
            ]
        )

print("Dependencies are available.")

Dependencies are available.


In [2]:
from pathlib import Path
from itertools import combinations, product
import gc
import hashlib
import json
import platform
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 350)
pd.set_option("display.max_rows", 350)
pd.set_option("display.max_colwidth", 300)

RANDOM_STATE_MODEL = 142
RANDOM_STATE_GENERAL = 42
PROBABILITY_EPSILON = 1e-12

np.random.seed(RANDOM_STATE_GENERAL)

In [3]:
RUN_MODE = "full_cv"  # "pilot", "full_cv", or "final_test"

ALLOW_OFFICIAL_TEST = False
# فقط برای اجرای نهایی:
# RUN_MODE = "final_test"
# ALLOW_OFFICIAL_TEST = True

ALL_FOLDS = [1, 2, 3, 4, 5]

ACTIVE_OUTER_FOLDS = (
    [1]
    if RUN_MODE == "pilot"
    else ALL_FOLDS
)

# Portable local paths. Override upstream data/artifacts with environment variables.
import os

def _find_repo_root():
    start = Path(os.environ.get("STRATAWEAVE_REPO_ROOT", Path.cwd())).expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("StrataWeave root not found. Open the repository in Jupyter or set STRATAWEAVE_REPO_ROOT.")

REPO_ROOT = _find_repo_root()
TRAIN_PATH = Path(os.environ.get("STRATAWEAVE_TRAIN_CSV", str(REPO_ROOT / "data" / "raw" / "train.csv"))).expanduser().resolve()
TEST_PATH = Path(os.environ.get("STRATAWEAVE_TEST_CSV", str(REPO_ROOT / "data" / "raw" / "test.csv"))).expanduser().resolve()
PROJECT_ROOT = Path(os.environ.get("STRATAWEAVE_ARTIFACT_ROOT", str(REPO_ROOT / "artifacts"))).expanduser().resolve()


FOLD_ASSIGNMENT_PATH = (
    PROJECT_ROOT
    / "01_clean_splits_support"
    / "10_grouped_5fold_well_assignment.csv"
)

STAGE_02A_DIR = (
    PROJECT_ROOT
    / "02A_legacy_RF_exact_reproduction"
)

LEGACY_TEST_SUMMARY_PATH = (
    STAGE_02A_DIR
    / "01_run_summary.csv"
)

LEGACY_REPRODUCTION_GATE_PATH = (
    STAGE_02A_DIR
    / "09_reproduction_gate.json"
)

LEGACY_OOF_SUMMARY_PATH = (
    PROJECT_ROOT
    / "02B_legacy_RF_grouped_well_CV"
    / "full"
    / "09_oof_summary.csv"
)

STAGE_ROOT = (
    PROJECT_ROOT
    / "07G_legacy_geo_stratigraphic_fusion"
)

OUTPUT_DIR = STAGE_ROOT / RUN_MODE

SHARED_CACHE_DIR = (
    STAGE_ROOT
    / "shared_evidence_cache"
)

FROZEN_SELECTION_PATH = (
    STAGE_ROOT
    / "full_cv"
    / "13_frozen_fusion_configuration.json"
)

FINAL_TEST_SENTINEL_PATH = (
    STAGE_ROOT
    / "final_test"
    / "00_OFFICIAL_TEST_CONSUMED.json"
)

for directory in [
    OUTPUT_DIR,
    SHARED_CACHE_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

required_common_paths = [
    TRAIN_PATH,
    FOLD_ASSIGNMENT_PATH,
    LEGACY_REPRODUCTION_GATE_PATH,
]

missing_common_paths = [
    str(path)
    for path in required_common_paths
    if not path.exists()
]

if missing_common_paths:
    raise FileNotFoundError(
        "Missing required inputs:\n"
        + "\n".join(
            missing_common_paths
        )
    )

if RUN_MODE == "final_test":
    if not ALLOW_OFFICIAL_TEST:
        raise RuntimeError(
            "final_test requested but ALLOW_OFFICIAL_TEST is False."
        )

    if not FROZEN_SELECTION_PATH.exists():
        raise FileNotFoundError(
            "Frozen full-CV configuration is missing:\n"
            + str(FROZEN_SELECTION_PATH)
        )

    if not TEST_PATH.exists():
        raise FileNotFoundError(TEST_PATH)

    if FINAL_TEST_SENTINEL_PATH.exists():
        raise RuntimeError(
            "Official test was already consumed by this stage."
        )

print("RUN_MODE:", RUN_MODE)
print("ACTIVE_OUTER_FOLDS:", ACTIVE_OUTER_FOLDS)
print("OUTPUT_DIR:", OUTPUT_DIR)

RUN_MODE: full_cv
ACTIVE_OUTER_FOLDS: [1, 2, 3, 4, 5]
OUTPUT_DIR: <LOCAL_PATH_REDACTED>


In [4]:
TARGET_COLUMN = (
    "FORCE_2020_LITHOFACIES_LITHOLOGY"
)

CONFIDENCE_COLUMN = (
    "FORCE_2020_LITHOFACIES_CONFIDENCE"
)

LITHOLOGY_CODES = sorted([
    30000,
    65030,
    65000,
    80000,
    74000,
    70000,
    70032,
    88000,
    86000,
    99000,
    90000,
    93000,
])

LITHOLOGY_NAMES = {
    30000: "Sandstone",
    65000: "Shale",
    65030: "Sandstone/Shale",
    70000: "Limestone",
    70032: "Chalk",
    74000: "Dolomite",
    80000: "Marl",
    86000: "Anhydrite",
    88000: "Halite",
    90000: "Coal",
    93000: "Basement",
    99000: "Tuff",
}

label_encoder = LabelEncoder()
label_encoder.fit(
    LITHOLOGY_CODES
)

CLASS_CODES = [
    int(value)
    for value in label_encoder.classes_
]

CLASS_INDICES = list(
    range(len(CLASS_CODES))
)

CLASS_INDEX_TO_CODE = {
    index: code
    for index, code
    in enumerate(CLASS_CODES)
}

CLASS_CODE_TO_INDEX = {
    code: index
    for index, code
    in CLASS_INDEX_TO_CODE.items()
}

CLASS_INDEX_TO_NAME = {
    index: LITHOLOGY_NAMES[code]
    for index, code
    in CLASS_INDEX_TO_CODE.items()
}

CHALK_INDEX = CLASS_CODE_TO_INDEX[70032]
SHALE_INDEX = CLASS_CODE_TO_INDEX[65000]

class_mapping = pd.DataFrame({
    "class_index": CLASS_INDICES,
    "lithology_code": CLASS_CODES,
    "lithology_name": [
        CLASS_INDEX_TO_NAME[index]
        for index in CLASS_INDICES
    ],
})

display(class_mapping)

,class_index,lithology_code,lithology_name
0,0,30000,Sandstone
1,1,65000,Shale
2,2,65030,Sandstone/Shale
3,3,70000,Limestone
4,4,70032,Chalk
5,5,74000,Dolomite
6,6,80000,Marl
7,7,86000,Anhydrite
8,8,88000,Halite
9,9,90000,Coal


In [5]:
TOP_LEGACY_FEATURES = [
    "GR_NPHI_interaction",
    "GR_DTC_interaction",
    "Y_LOC",
    "GR_RHOB_RATIO",
    "NPHI_DTC_interaction",
    "X_LOC",
    "RHOB_DTC_interaction",
    "CALI",
    "Z_LOC",
    "GR_RHOB_interaction",
    "DEPTH_NORMALIZED",
    "DEPTH_MD",
    "RDEP",
    "RHOB_log",
    "RMED",
    "RHOB",
    "X_LOC",  # تکرار عمدی؛ مدل تغییر نمی‌کند
]

EXTRA_WEAK_CLASS_FEATURES = [
    "GR_flatness",
    "DTC_grad",
    "DTC_RHOB_DIFF",
    "DTC_RHOB_RATIO2",
    "RDEP_GR_RATIO",
    "RHOB_NPHI_DIFF2",
    "Acoustic_Impedance",
    "Sand_indicator",
    "Shale_indicator",
]

LEGACY_FEATURE_COLUMNS = (
    TOP_LEGACY_FEATURES
    + EXTRA_WEAK_CLASS_FEATURES
)

assert len(
    LEGACY_FEATURE_COLUMNS
) == 26

assert len(
    set(LEGACY_FEATURE_COLUMNS)
) == 25

assert (
    LEGACY_FEATURE_COLUMNS.count(
        "X_LOC"
    )
    == 2
)

print("Feature columns:", len(LEGACY_FEATURE_COLUMNS))
print("Unique names:", len(set(LEGACY_FEATURE_COLUMNS)))
print(LEGACY_FEATURE_COLUMNS)

Feature columns: 26
Unique names: 25
['GR_NPHI_interaction', 'GR_DTC_interaction', 'Y_LOC', 'GR_RHOB_RATIO', 'NPHI_DTC_interaction', 'X_LOC', 'RHOB_DTC_interaction', 'CALI', 'Z_LOC', 'GR_RHOB_interaction', 'DEPTH_NORMALIZED', 'DEPTH_MD', 'RDEP', 'RHOB_log', 'RMED', 'RHOB', 'X_LOC', 'GR_flatness', 'DTC_grad', 'DTC_RHOB_DIFF', 'DTC_RHOB_RATIO2', 'RDEP_GR_RATIO', 'RHOB_NPHI_DIFF2', 'Acoustic_Impedance', 'Sand_indicator', 'Shale_indicator']


In [6]:
LEGACY_EVIDENCE_FEATURES = [
    feature
    for feature in LEGACY_FEATURE_COLUMNS
    if feature not in {
        "X_LOC",
        "Y_LOC",
    }
]

assert "X_LOC" not in LEGACY_EVIDENCE_FEATURES
assert "Y_LOC" not in LEGACY_EVIDENCE_FEATURES
assert len(LEGACY_EVIDENCE_FEATURES) == 23

REQUIRED_CAUSAL_ENGINEERED_FEATURES = {
    "GR_NPHI_interaction",
    "GR_DTC_interaction",
    "GR_RHOB_RATIO",
    "NPHI_DTC_interaction",
    "RHOB_DTC_interaction",
    "GR_RHOB_interaction",
    "RHOB_log",
    "DTC_RHOB_DIFF",
    "DTC_RHOB_RATIO2",
    "RDEP_GR_RATIO",
    "RHOB_NPHI_DIFF2",
    "Acoustic_Impedance",
    "Sand_indicator",
    "Shale_indicator",
}

assert REQUIRED_CAUSAL_ENGINEERED_FEATURES.issubset(
    set(LEGACY_EVIDENCE_FEATURES)
)

RF_PARAMETERS = {
    "n_estimators": 800,
    "max_depth": 200,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "random_state": RANDOM_STATE_MODEL,
    "n_jobs": -1,
    "verbose": 1,
}

feature_registry = pd.DataFrame({
    "legacy_position": list(
        range(len(LEGACY_FEATURE_COLUMNS))
    ),
    "feature": LEGACY_FEATURE_COLUMNS,
    "used_in_evidence_RF": [
        feature not in {
            "X_LOC",
            "Y_LOC",
        }
        for feature in LEGACY_FEATURE_COLUMNS
    ],
    "moved_to_geo_prior_branch": [
        feature in {
            "X_LOC",
            "Y_LOC",
        }
        for feature in LEGACY_FEATURE_COLUMNS
    ],
})

display(feature_registry)

,legacy_position,feature,used_in_evidence_RF,moved_to_geo_prior_branch
0,0,GR_NPHI_interaction,True,False
1,1,GR_DTC_interaction,True,False
2,2,Y_LOC,False,True
3,3,GR_RHOB_RATIO,True,False
4,4,NPHI_DTC_interaction,True,False
5,5,X_LOC,False,True
6,6,RHOB_DTC_interaction,True,False
7,7,CALI,True,False
8,8,Z_LOC,True,False
9,9,GR_RHOB_interaction,True,False


# 3. Fusion candidate contract

In [7]:
K_NEIGHBORS_GRID = [
    3,
    5,
]

VERTICAL_HALF_WINDOW_GRID = [
    15.0,
    30.0,
    60.0,
]

LAMBDA_MAX_GRID = [
    0.25,
    0.50,
    0.75,
]

SPATIAL_RATIO_CLIP_GRID = [
    1.5,
    2.0,
]

RESAMPLING_PRIOR_CORRECTION_POWER_GRID = [
    0.0,
    0.5,
    1.0,
]

USE_FORMATION_HIERARCHY = True

MIN_ROWS_PER_WELL_WINDOW = 5
MIN_FORMATION_SUPPORT_WELLS = 2
MIN_WELLS_FULL_TRUST = 4
ROWS_PER_WELL_FULL_TRUST = 20.0
PRIOR_SMOOTHING_EQUIVALENT_WELLS = 2.0

candidate_rows = []

for (
    k_neighbors,
    vertical_half_window,
    lambda_max,
    spatial_ratio_clip,
    resampling_prior_power,
) in product(
    K_NEIGHBORS_GRID,
    VERTICAL_HALF_WINDOW_GRID,
    LAMBDA_MAX_GRID,
    SPATIAL_RATIO_CLIP_GRID,
    RESAMPLING_PRIOR_CORRECTION_POWER_GRID,
):
    candidate_id = (
        f"k{k_neighbors}"
        f"__z{vertical_half_window:g}"
        f"__lam{lambda_max:g}"
        f"__clip{spatial_ratio_clip:g}"
        f"__priorcorr{resampling_prior_power:g}"
    )

    candidate_rows.append({
        "candidate_id": candidate_id,
        "k_neighbors": int(k_neighbors),
        "vertical_half_window": float(
            vertical_half_window
        ),
        "lambda_max": float(lambda_max),
        "spatial_ratio_clip": float(
            spatial_ratio_clip
        ),
        "resampling_prior_correction_power": float(
            resampling_prior_power
        ),
        "formation_hierarchy": bool(
            USE_FORMATION_HIERARCHY
        ),
        "well_balanced_prior": True,
        "adaptive_support_weight": True,
        "adaptive_distance_weight": True,
        "adaptive_depth_weight": True,
        "adaptive_agreement_weight": True,
        "global_fallback": True,
    })

candidate_registry = pd.DataFrame(
    candidate_rows
)

print(
    "Fusion candidates:",
    len(candidate_registry),
)

display(
    candidate_registry.head(20)
)

Fusion candidates: 108


,candidate_id,k_neighbors,vertical_half_window,lambda_max,spatial_ratio_clip,resampling_prior_correction_power,formation_hierarchy,well_balanced_prior,adaptive_support_weight,adaptive_distance_weight,adaptive_depth_weight,adaptive_agreement_weight,global_fallback
0,k3__z15__lam0.25__clip1.5__priorcorr0,3,15.0,0.25,1.5,0.0,True,True,True,True,True,True,True
1,k3__z15__lam0.25__clip1.5__priorcorr0.5,3,15.0,0.25,1.5,0.5,True,True,True,True,True,True,True
2,k3__z15__lam0.25__clip1.5__priorcorr1,3,15.0,0.25,1.5,1.0,True,True,True,True,True,True,True
3,k3__z15__lam0.25__clip2__priorcorr0,3,15.0,0.25,2.0,0.0,True,True,True,True,True,True,True
4,k3__z15__lam0.25__clip2__priorcorr0.5,3,15.0,0.25,2.0,0.5,True,True,True,True,True,True,True
5,k3__z15__lam0.25__clip2__priorcorr1,3,15.0,0.25,2.0,1.0,True,True,True,True,True,True,True
6,k3__z15__lam0.5__clip1.5__priorcorr0,3,15.0,0.50,1.5,0.0,True,True,True,True,True,True,True
7,k3__z15__lam0.5__clip1.5__priorcorr0.5,3,15.0,0.50,1.5,0.5,True,True,True,True,True,True,True
8,k3__z15__lam0.5__clip1.5__priorcorr1,3,15.0,0.50,1.5,1.0,True,True,True,True,True,True,True
9,k3__z15__lam0.5__clip2__priorcorr0,3,15.0,0.50,2.0,0.0,True,True,True,True,True,True,True


In [8]:
def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, Path):
        return str(value)

    if (
        isinstance(value, float)
        and not np.isfinite(value)
    ):
        return None

    return value


def detect_csv_format(
    file_path,
    num_lines=5,
):
    for encoding in [
        "utf-8",
        "latin1",
        "iso-8859-1",
        "cp1252",
    ]:
        try:
            with open(
                file_path,
                "r",
                encoding=encoding,
            ) as file:
                lines = [
                    file.readline()
                    for _ in range(
                        num_lines
                    )
                ]

            if any(
                "," in line
                for line in lines
            ):
                separator = ","

            elif any(
                ";" in line
                for line in lines
            ):
                separator = ";"

            elif any(
                "\t" in line
                for line in lines
            ):
                separator = "\t"

            else:
                separator = ","

            return {
                "sep": separator,
                "encoding": encoding,
            }

        except UnicodeDecodeError:
            continue

    return {
        "sep": ",",
        "encoding": "utf-8",
    }


def load_csv_full(file_path):
    csv_format = detect_csv_format(
        file_path
    )

    print(
        f"Loading {file_path.name} "
        f"with {csv_format}"
    )

    return pd.read_csv(
        file_path,
        sep=csv_format["sep"],
        encoding=csv_format[
            "encoding"
        ],
        low_memory=False,
    )


def normalize_probability_rows(matrix):
    matrix = np.asarray(
        matrix,
        dtype=np.float64,
    )

    matrix = np.clip(
        matrix,
        PROBABILITY_EPSILON,
        np.inf,
    )

    row_sums = matrix.sum(
        axis=1,
        keepdims=True,
    )

    return matrix / row_sums


def softmax_from_log_scores(
    log_scores,
):
    log_scores = np.asarray(
        log_scores,
        dtype=np.float64,
    )

    shifted = (
        log_scores
        - log_scores.max(
            axis=1,
            keepdims=True,
        )
    )

    return normalize_probability_rows(
        np.exp(shifted)
    )


def align_probabilities(
    raw_probabilities,
    trained_classes,
):
    aligned = np.zeros(
        (
            raw_probabilities.shape[0],
            len(CLASS_INDICES),
        ),
        dtype=np.float64,
    )

    for source_column, class_index in enumerate(
        trained_classes
    ):
        aligned[
            :,
            int(class_index),
        ] = raw_probabilities[
            :,
            source_column,
        ]

    return normalize_probability_rows(
        aligned
    )


def calculate_metrics(
    y_true,
    y_pred,
    probabilities=None,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    present_labels = sorted(
        np.unique(y_true).tolist()
    )

    metrics = {
        "rows": int(len(y_true)),
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_precision_present_classes": float(
            precision_score(
                y_true,
                y_pred,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall_present_classes": float(
            recall_score(
                y_true,
                y_pred,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1_present_classes": float(
            f1_score(
                y_true,
                y_pred,
                labels=present_labels,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1_all_12_classes": float(
            f1_score(
                y_true,
                y_pred,
                labels=CLASS_INDICES,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
    }

    if probabilities is not None:
        metrics["log_loss"] = float(
            log_loss(
                y_true,
                probabilities,
                labels=CLASS_INDICES,
            )
        )

    return metrics


def calculate_per_class_metrics(
    y_true,
    y_pred,
):
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=CLASS_INDICES,
            zero_division=0,
        )
    )

    return pd.DataFrame({
        "class_index": CLASS_INDICES,
        "lithology_code": CLASS_CODES,
        "lithology_name": [
            CLASS_INDEX_TO_NAME[index]
            for index in CLASS_INDICES
        ],
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })


def apply_legacy_chalk_refinement(
    probabilities,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    raw_predictions = np.argmax(
        probabilities,
        axis=1,
    ).astype(int)

    probability_order = np.argsort(
        probabilities,
        axis=1,
    )[:, ::-1]

    second_class = probability_order[
        :,
        1
    ]

    refined_predictions = (
        raw_predictions.copy()
    )

    refinement_mask = (
        (
            second_class
            == CHALK_INDEX
        )
        & (
            raw_predictions
            != CHALK_INDEX
        )
    )

    refined_predictions[
        refinement_mask
    ] = CHALK_INDEX

    return (
        raw_predictions,
        refined_predictions,
        refinement_mask,
    )


def calculate_per_well_metrics(
    metadata,
    probabilities,
    raw_predictions,
    refined_predictions,
):
    rows = []

    for well_name, positions_index in (
        metadata.groupby(
            "WELL"
        ).groups.items()
    ):
        positions = np.asarray(
            list(positions_index),
            dtype=int,
        )

        y_true = metadata.loc[
            positions,
            "LITHOLOGY_ENCODED",
        ].to_numpy(
            dtype=int
        )

        present_labels = sorted(
            np.unique(y_true).tolist()
        )

        rows.append({
            "WELL": well_name,
            "row_count": int(
                len(positions)
            ),
            "class_count": int(
                len(present_labels)
            ),
            "raw_accuracy": float(
                accuracy_score(
                    y_true,
                    raw_predictions[
                        positions
                    ],
                )
            ),
            "refined_accuracy": float(
                accuracy_score(
                    y_true,
                    refined_predictions[
                        positions
                    ],
                )
            ),
            "raw_balanced_accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    raw_predictions[
                        positions
                    ],
                )
            ),
            "refined_balanced_accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    refined_predictions[
                        positions
                    ],
                )
            ),
            "raw_macro_f1_present_classes": float(
                f1_score(
                    y_true,
                    raw_predictions[
                        positions
                    ],
                    labels=present_labels,
                    average="macro",
                    zero_division=0,
                )
            ),
            "refined_macro_f1_present_classes": float(
                f1_score(
                    y_true,
                    refined_predictions[
                        positions
                    ],
                    labels=present_labels,
                    average="macro",
                    zero_division=0,
                )
            ),
            "log_loss": float(
                log_loss(
                    y_true,
                    probabilities[
                        positions
                    ],
                    labels=CLASS_INDICES,
                )
            ),
        })

    return pd.DataFrame(rows)

In [9]:
def fit_and_apply_train_imputers(
    dataframe,
):
    df = dataframe.copy()

    if CONFIDENCE_COLUMN in df.columns:
        df = df.drop(
            columns=[CONFIDENCE_COLUMN]
        )

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    for excluded in [
        TARGET_COLUMN,
        "LITHOLOGY_ENCODED",
        "GROUP_FOLD",
    ]:
        if excluded in numeric_columns:
            numeric_columns.remove(
                excluded
            )

    imputers = {}

    for column in numeric_columns:
        if df[column].isna().any():
            column_imputer = (
                SimpleImputer(
                    strategy="median"
                )
            )

            df[column] = (
                column_imputer
                .fit_transform(
                    df[[column]]
                )
                .ravel()
            )

            imputers[column] = (
                column_imputer
            )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return (
        df,
        imputers,
    )


def apply_train_imputers_to_validation(
    dataframe,
    imputers,
):
    df = dataframe.copy()

    if CONFIDENCE_COLUMN in df.columns:
        df = df.drop(
            columns=[CONFIDENCE_COLUMN]
        )

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    for excluded in [
        TARGET_COLUMN,
        "LITHOLOGY_ENCODED",
        "GROUP_FOLD",
    ]:
        if excluded in numeric_columns:
            numeric_columns.remove(
                excluded
            )

    for column in numeric_columns:
        if (
            column in imputers
            and df[column].isna().any()
        ):
            df[column] = (
                imputers[column]
                .transform(
                    df[[column]]
                )
                .ravel()
            )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df


def add_legacy_preprocess_features(
    dataframe,
):
    df = dataframe.copy()
    epsilon = 1e-10

    if {
        "NPHI",
        "RHOB",
    }.issubset(df.columns):
        df[
            "NPHI_RHOB_RATIO"
        ] = (
            df["NPHI"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if {
        "GR",
        "RHOB",
    }.issubset(df.columns):
        df[
            "GR_RHOB_RATIO"
        ] = (
            df["GR"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if {
        "DTC",
        "RHOB",
    }.issubset(df.columns):
        df[
            "DTC_RHOB_RATIO"
        ] = (
            df["DTC"]
            / (
                df["RHOB"].abs()
                + epsilon
            )
        )

    if "DEPTH_MD" in df.columns:
        depth_min = df[
            "DEPTH_MD"
        ].min()

        depth_max = df[
            "DEPTH_MD"
        ].max()

        depth_range = (
            depth_max
            - depth_min
        )

        if depth_range > 0:
            df[
                "DEPTH_NORMALIZED"
            ] = (
                (
                    df["DEPTH_MD"]
                    - depth_min
                )
                / depth_range
            )
        else:
            df[
                "DEPTH_NORMALIZED"
            ] = 0.0

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
    )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df


def encode_target(
    dataframe,
):
    df = dataframe.copy()

    numeric_target = (
        pd.to_numeric(
            df[TARGET_COLUMN],
            errors="raise",
        )
        .astype(int)
    )

    unknown_codes = sorted(
        set(numeric_target)
        - set(CLASS_CODES)
    )

    if unknown_codes:
        raise ValueError(
            f"Unknown lithology codes: {unknown_codes}"
        )

    df[TARGET_COLUMN] = (
        numeric_target
    )

    df[
        "LITHOLOGY_ENCODED"
    ] = label_encoder.transform(
        numeric_target
    )

    return df


def apply_legacy_exact_resampling(
    dataframe,
):
    df = dataframe.copy()

    before_counts = (
        df[TARGET_COLUMN]
        .value_counts()
        .sort_index()
    )

    def drop_fraction(
        source_df,
        class_code,
        fraction,
        random_state,
    ):
        class_mask = (
            source_df[
                TARGET_COLUMN
            ]
            == class_code
        )

        class_count = int(
            class_mask.sum()
        )

        if class_count == 0:
            return source_df

        n_to_drop = int(
            class_count
            * fraction
        )

        indices_to_drop = (
            source_df.loc[
                class_mask
            ]
            .sample(
                n=n_to_drop,
                random_state=random_state,
            )
            .index
        )

        return source_df.drop(
            indices_to_drop
        )

    df = drop_fraction(
        df,
        class_code=65000,
        fraction=0.30,
        random_state=86,
    )

    df = drop_fraction(
        df,
        class_code=30000,
        fraction=0.20,
        random_state=86,
    )

    df = drop_fraction(
        df,
        class_code=70000,
        fraction=0.20,
        random_state=142,
    )

    df = drop_fraction(
        df,
        class_code=65030,
        fraction=0.20,
        random_state=142,
    )

    # رفتار واقعی کد مقاله قبلی حفظ می‌شود.
    basement_mask = (
        df[TARGET_COLUMN]
        == 93000
    )

    dolomite_mask = (
        df[TARGET_COLUMN]
        == 74000
    )

    dolomite_count = int(
        dolomite_mask.sum()
    )

    if (
        dolomite_count > 0
        and basement_mask.sum() > 0
    ):
        dolomite_target_count = int(
            dolomite_count
            * 1.5
        )

        n_to_add = (
            dolomite_target_count
            - dolomite_count
        )

        samples_to_add = (
            df.loc[
                basement_mask
            ]
            .sample(
                n=n_to_add,
                replace=True,
                random_state=42,
            )
        )

        df = pd.concat(
            [
                df,
                samples_to_add,
            ],
            ignore_index=True,
        )

    after_counts = (
        df[TARGET_COLUMN]
        .value_counts()
        .sort_index()
    )

    audit_rows = []

    for class_code in CLASS_CODES:
        before = int(
            before_counts.get(
                class_code,
                0,
            )
        )

        after = int(
            after_counts.get(
                class_code,
                0,
            )
        )

        audit_rows.append({
            "lithology_code": class_code,
            "lithology_name": (
                LITHOLOGY_NAMES[
                    class_code
                ]
            ),
            "before_count": before,
            "after_count": after,
            "change": after - before,
            "retained_fraction": (
                after / before
                if before > 0
                else np.nan
            ),
        })

    return (
        df,
        pd.DataFrame(
            audit_rows
        ),
    )

In [10]:
BASE_CAUSAL_FEATURES = [
    "GR",
    "NPHI",
    "RHOB",
    "PEF",
    "DTC",
    "SP",
]

def compute_legacy_causal_features(
    dataframe,
):
    df = dataframe.copy()

    available = [
        feature
        for feature in BASE_CAUSAL_FEATURES
        if feature in df.columns
    ]

    for feature_1, feature_2 in combinations(
        available,
        2,
    ):
        df[
            f"{feature_1}_{feature_2}_interaction"
        ] = (
            df[feature_1]
            * df[feature_2]
        )

    for feature in available:
        df[
            f"{feature}_squared"
        ] = (
            df[feature] ** 2
        )

        df[
            f"{feature}_log"
        ] = np.log(
            np.abs(
                df[feature]
            )
            + 1
        )

    return df


def add_legacy_weak_class_features(
    dataframe,
):
    df = dataframe.copy()
    epsilon = 1e-6

    def has_columns(
        columns,
    ):
        return all(
            column in df.columns
            for column in columns
        )

    # رفتار Legacy:
    # Train بدون WELL؛ Validation با WELL
    if has_columns(["GR"]):
        if "WELL" in df.columns:
            df["GR_grad"] = (
                df.groupby(
                    "WELL"
                )["GR"]
                .diff()
                .fillna(0)
            )

            df["GR_flatness"] = (
                df.groupby(
                    "WELL"
                )["GR"]
                .rolling(
                    window=5,
                    min_periods=1,
                )
                .std()
                .reset_index(
                    level=0,
                    drop=True,
                )
            )
        else:
            df["GR_grad"] = (
                df["GR"]
                .diff()
                .fillna(0)
            )

            df["GR_flatness"] = (
                df["GR"]
                .rolling(
                    5,
                    min_periods=1,
                )
                .std()
            )

    if has_columns(["DTC"]):
        if "WELL" in df.columns:
            df["DTC_grad"] = (
                df.groupby(
                    "WELL"
                )["DTC"]
                .diff()
                .fillna(0)
            )
        else:
            df["DTC_grad"] = (
                df["DTC"]
                .diff()
                .fillna(0)
            )

    if has_columns(["RDEP"]):
        if "WELL" in df.columns:
            df["RDEP_grad"] = (
                df.groupby(
                    "WELL"
                )["RDEP"]
                .diff()
                .fillna(0)
            )
        else:
            df["RDEP_grad"] = (
                df["RDEP"]
                .diff()
                .fillna(0)
            )

    if has_columns([
        "DTC",
        "RHOB",
    ]):
        df[
            "DTC_RHOB_DIFF"
        ] = (
            df["DTC"]
            - df["RHOB"]
        )

        df[
            "DTC_RHOB_RATIO2"
        ] = (
            df["DTC"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if has_columns([
        "RDEP",
        "GR",
    ]):
        df[
            "RDEP_GR_RATIO"
        ] = (
            df["RDEP"]
            / (
                df["GR"]
                + epsilon
            )
        )

    if has_columns([
        "RHOB",
        "NPHI",
    ]):
        df[
            "RHOB_NPHI_DIFF2"
        ] = (
            df["RHOB"]
            - df["NPHI"]
        )

    if has_columns(["GR"]):
        gr_min = df["GR"].min()
        gr_max = df["GR"].max()

        df[
            "VSH_linear"
        ] = (
            (
                df["GR"]
                - gr_min
            )
            / (
                gr_max
                - gr_min
                + epsilon
            )
        )

        df[
            "VSH_Larionov"
        ] = (
            0.083
            * (
                2 ** (
                    3.7
                    * df[
                        "VSH_linear"
                    ]
                )
                - 1
            )
        )

        df[
            "VSH_Stieber"
        ] = (
            df["VSH_linear"]
            / (
                3
                - 2
                * df[
                    "VSH_linear"
                ]
                + epsilon
            )
        )

    if has_columns(["DTC"]):
        df["Vp"] = (
            1e6
            / (
                df["DTC"]
                + epsilon
            )
        )

    if has_columns([
        "RHOB",
        "Vp",
    ]):
        df[
            "Acoustic_Impedance"
        ] = (
            df["RHOB"]
            * df["Vp"]
        )

    if has_columns([
        "NPHI",
        "RHOB",
    ]):
        df[
            "Poisson_proxy"
        ] = (
            df["NPHI"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if "NPHI" in df.columns:
        df["Brittleness"] = (
            1
            - df["NPHI"]
        )

    if has_columns([
        "Vp",
        "RHOB",
    ]):
        df[
            "Sand_indicator"
        ] = (
            df["Vp"]
            / (
                df["RHOB"]
                + epsilon
            )
        )

    if has_columns([
        "RHOB",
        "NPHI",
        "VSH_Larionov",
    ]):
        df[
            "Shale_indicator"
        ] = (
            df["RHOB"]
            * df["NPHI"]
            * df[
                "VSH_Larionov"
            ]
        )

    numeric_columns = (
        df.select_dtypes(
            include=[np.number]
        )
        .columns
    )

    df[numeric_columns] = (
        df[numeric_columns]
        .fillna(0)
        .replace(
            [np.inf, -np.inf],
            0,
        )
    )

    return df

# 6. Evidence Random Forest pipeline

In [11]:
def class_prior_from_codes(
    class_codes,
):
    counts = (
        pd.Series(
            np.asarray(
                class_codes,
                dtype=int,
            )
        )
        .value_counts()
        .reindex(
            CLASS_CODES,
            fill_value=0,
        )
        .to_numpy(
            dtype=np.float64
        )
    )

    counts += PROBABILITY_EPSILON

    return counts / counts.sum()


def prepare_evidence_train_and_evaluation(
    raw_training,
    raw_evaluation,
):
    raw_training = (
        raw_training.copy()
        .reset_index(drop=True)
    )

    raw_evaluation = (
        raw_evaluation.copy()
        .reset_index(drop=True)
    )

    raw_training_prior = (
        class_prior_from_codes(
            pd.to_numeric(
                raw_training[
                    TARGET_COLUMN
                ],
                errors="raise",
            )
            .astype(int)
            .to_numpy()
        )
    )

    (
        train_imputed,
        train_imputers,
    ) = fit_and_apply_train_imputers(
        raw_training
    )

    train_preprocessed = (
        add_legacy_preprocess_features(
            train_imputed
        )
    )

    (
        train_resampled,
        resampling_audit,
    ) = apply_legacy_exact_resampling(
        train_preprocessed
    )

    resampled_training_prior = (
        class_prior_from_codes(
            train_resampled[
                TARGET_COLUMN
            ].to_numpy(
                dtype=int
            )
        )
    )

    train_resampled = encode_target(
        train_resampled
    )

    train_causal = (
        compute_legacy_causal_features(
            train_resampled
        )
    )

    exclude_columns = {
        "WELL",
        TARGET_COLUMN,
        CONFIDENCE_COLUMN,
        "LITHOLOGY_ENCODED",
        "GROUP",
        "FORMATION",
        "ORIGINAL_ROW_ID",
    }

    candidate_numeric_features = [
        column
        for column in train_causal.columns
        if (
            column not in exclude_columns
            and pd.api.types.is_numeric_dtype(
                train_causal[column]
            )
        )
    ]

    # Exact Legacy asymmetry:
    # Training weak features are built without WELL.
    train_numeric_only = (
        train_causal[
            candidate_numeric_features
        ]
        .copy()
    )

    train_with_weak = (
        add_legacy_weak_class_features(
            train_numeric_only
        )
    )

    missing_train_features = [
        feature
        for feature in set(
            LEGACY_EVIDENCE_FEATURES
        )
        if feature not in (
            train_with_weak.columns
        )
    ]

    if missing_train_features:
        raise RuntimeError(
            "Missing evidence Train features: "
            + str(
                missing_train_features
            )
        )

    X_train = (
        train_with_weak[
            LEGACY_EVIDENCE_FEATURES
        ]
        .copy()
    )

    y_train = (
        train_causal[
            "LITHOLOGY_ENCODED"
        ]
        .astype(int)
        .to_numpy()
    )

    evaluation_imputed = (
        apply_train_imputers_to_validation(
            raw_evaluation,
            imputers=train_imputers,
        )
    )

    evaluation_preprocessed = (
        add_legacy_preprocess_features(
            evaluation_imputed
        )
    )

    evaluation_preprocessed = (
        encode_target(
            evaluation_preprocessed
        )
    )

    evaluation_causal = (
        compute_legacy_causal_features(
            evaluation_preprocessed
        )
    )

    # Exact Legacy asymmetry:
    # Evaluation weak features retain WELL.
    evaluation_with_weak = (
        add_legacy_weak_class_features(
            evaluation_causal
        )
    )

    missing_evaluation_features = [
        feature
        for feature in set(
            LEGACY_EVIDENCE_FEATURES
        )
        if feature not in (
            evaluation_with_weak.columns
        )
    ]

    if missing_evaluation_features:
        raise RuntimeError(
            "Missing evidence Evaluation features: "
            + str(
                missing_evaluation_features
            )
        )

    X_evaluation = (
        evaluation_with_weak[
            LEGACY_EVIDENCE_FEATURES
        ]
        .copy()
    )

    y_evaluation = (
        evaluation_with_weak[
            "LITHOLOGY_ENCODED"
        ]
        .astype(int)
        .to_numpy()
    )

    metadata_columns = [
        "WELL",
        "DEPTH_MD",
        TARGET_COLUMN,
        "LITHOLOGY_ENCODED",
        "ORIGINAL_ROW_ID",
    ]

    for optional_column in [
        "X_LOC",
        "Y_LOC",
        "Z_LOC",
        "FORMATION",
    ]:
        if optional_column in (
            evaluation_with_weak.columns
        ):
            metadata_columns.append(
                optional_column
            )

    evaluation_metadata = (
        evaluation_with_weak[
            metadata_columns
        ]
        .copy()
        .reset_index(drop=True)
    )

    scaler = StandardScaler()

    X_train_scaled = (
        scaler.fit_transform(
            X_train
        )
    )

    X_evaluation_scaled = (
        scaler.transform(
            X_evaluation
        )
    )

    return {
        "X_train_scaled": X_train_scaled,
        "y_train": y_train,
        "X_evaluation_scaled": (
            X_evaluation_scaled
        ),
        "y_evaluation": y_evaluation,
        "evaluation_metadata": (
            evaluation_metadata
        ),
        "train_imputers": train_imputers,
        "scaler": scaler,
        "resampling_audit": (
            resampling_audit
        ),
        "raw_training_prior": (
            raw_training_prior
        ),
        "resampled_training_prior": (
            resampled_training_prior
        ),
        "raw_training_rows": int(
            len(raw_training)
        ),
        "resampled_training_rows": int(
            len(X_train)
        ),
    }


def fit_evidence_RF(prepared):
    model = RandomForestClassifier(
        **RF_PARAMETERS
    )

    fit_start = time.time()

    model.fit(
        prepared[
            "X_train_scaled"
        ],
        prepared[
            "y_train"
        ],
    )

    fit_seconds = (
        time.time()
        - fit_start
    )

    probability_partial = (
        model.predict_proba(
            prepared[
                "X_evaluation_scaled"
            ]
        )
    )

    evidence_probabilities = (
        align_probabilities(
            probability_partial,
            model.classes_,
        )
    )

    return {
        "model": model,
        "probabilities": (
            evidence_probabilities
        ),
        "fit_seconds": float(
            fit_seconds
        ),
    }


def apply_resampling_prior_correction(
    evidence_probabilities,
    raw_training_prior,
    resampled_training_prior,
    power,
):
    ratio = (
        np.clip(
            raw_training_prior,
            PROBABILITY_EPSILON,
            1.0,
        )
        / np.clip(
            resampled_training_prior,
            PROBABILITY_EPSILON,
            1.0,
        )
    )

    return normalize_probability_rows(
        np.asarray(
            evidence_probabilities,
            dtype=np.float64,
        )
        * (
            ratio
            ** float(power)
        )[
            None,
            :
        ]
    )

# 7. Geo-stratigraphic prior bank

In [12]:
def clean_numeric_series(series):
    values = pd.to_numeric(
        series,
        errors="coerce",
    ).astype(float)

    values.loc[
        ~np.isfinite(
            values.to_numpy(
                dtype=float
            )
        )
    ] = np.nan

    return values


def choose_vertical_column(
    training_dataframe,
):
    for column in [
        "Z_LOC",
        "DEPTH_MD",
    ]:
        if column not in (
            training_dataframe.columns
        ):
            continue

        values = clean_numeric_series(
            training_dataframe[
                column
            ]
        )

        coverage = float(
            values.notna().mean()
        )

        unique_count = int(
            values.nunique(
                dropna=True
            )
        )

        if (
            coverage >= 0.80
            and unique_count >= 100
        ):
            return {
                "vertical_column": column,
                "coverage": coverage,
                "unique_count": unique_count,
            }

    raise RuntimeError(
        "Neither Z_LOC nor DEPTH_MD has sufficient support."
    )


def create_prefix_profile(
    z_values,
    encoded_labels,
):
    z_values = np.asarray(
        z_values,
        dtype=np.float64,
    )

    encoded_labels = np.asarray(
        encoded_labels,
        dtype=np.int16,
    )

    order = np.argsort(
        z_values,
        kind="mergesort",
    )

    z_sorted = z_values[
        order
    ]

    labels_sorted = encoded_labels[
        order
    ]

    one_hot = np.zeros(
        (
            len(z_sorted),
            len(CLASS_INDICES),
        ),
        dtype=np.int32,
    )

    one_hot[
        np.arange(
            len(z_sorted)
        ),
        labels_sorted,
    ] = 1

    prefix = np.zeros(
        (
            len(z_sorted) + 1,
            len(CLASS_INDICES),
        ),
        dtype=np.int32,
    )

    prefix[
        1:
    ] = np.cumsum(
        one_hot,
        axis=0,
        dtype=np.int32,
    )

    return {
        "z": z_sorted,
        "prefix": prefix,
        "rows": int(len(z_sorted)),
    }


def counts_from_profile(
    profile,
    query_z,
    half_window,
):
    query_z = np.asarray(
        query_z,
        dtype=np.float64,
    )

    z_sorted = profile[
        "z"
    ]

    prefix = profile[
        "prefix"
    ]

    left = np.searchsorted(
        z_sorted,
        query_z
        - half_window,
        side="left",
    )

    right = np.searchsorted(
        z_sorted,
        query_z
        + half_window,
        side="right",
    )

    return (
        prefix[
            right
        ]
        - prefix[
            left
        ]
    ).astype(
        np.float64,
        copy=False,
    )


def build_geo_prior_bank(
    raw_training,
):
    vertical_info = (
        choose_vertical_column(
            raw_training
        )
    )

    vertical_column = (
        vertical_info[
            "vertical_column"
        ]
    )

    required_columns = {
        "WELL",
        "X_LOC",
        "Y_LOC",
        vertical_column,
        TARGET_COLUMN,
    }

    missing_columns = (
        required_columns
        - set(raw_training.columns)
    )

    if missing_columns:
        raise KeyError(
            "Missing geo-prior columns: "
            + str(
                sorted(
                    missing_columns
                )
            )
        )

    work = raw_training.copy()

    work[
        "X_LOC"
    ] = clean_numeric_series(
        work[
            "X_LOC"
        ]
    )

    work[
        "Y_LOC"
    ] = clean_numeric_series(
        work[
            "Y_LOC"
        ]
    )

    work[
        vertical_column
    ] = clean_numeric_series(
        work[
            vertical_column
        ]
    )

    work[
        TARGET_COLUMN
    ] = pd.to_numeric(
        work[
            TARGET_COLUMN
        ],
        errors="raise",
    ).astype(int)

    work[
        "LITHOLOGY_ENCODED"
    ] = label_encoder.transform(
        work[
            TARGET_COLUMN
        ]
    )

    coordinate_table = (
        work.groupby(
            "WELL",
            as_index=False,
        )
        .agg(
            X_LOC=(
                "X_LOC",
                "median",
            ),
            Y_LOC=(
                "Y_LOC",
                "median",
            ),
        )
    )

    coordinate_table = (
        coordinate_table.loc[
            coordinate_table[
                [
                    "X_LOC",
                    "Y_LOC",
                ]
            ]
            .notna()
            .all(axis=1)
        ]
        .reset_index(drop=True)
    )

    profile_work = (
        work.loc[
            work[
                vertical_column
            ].notna()
        ]
        .copy()
    )

    all_profiles = {}

    for well_name, well_frame in (
        profile_work.groupby(
            "WELL",
            sort=False,
        )
    ):
        all_profiles[
            well_name
        ] = create_prefix_profile(
            well_frame[
                vertical_column
            ].to_numpy(
                dtype=float
            ),
            well_frame[
                "LITHOLOGY_ENCODED"
            ].to_numpy(
                dtype=int
            ),
        )

    formation_profiles = {}

    formation_available = bool(
        USE_FORMATION_HIERARCHY
        and "FORMATION" in (
            profile_work.columns
        )
    )

    if formation_available:
        formation_work = (
            profile_work.loc[
                profile_work[
                    "FORMATION"
                ].notna()
            ]
            .copy()
        )

        formation_work[
            "FORMATION_KEY"
        ] = (
            formation_work[
                "FORMATION"
            ]
            .astype(str)
        )

        for (
            well_name,
            formation_key,
        ), group_frame in (
            formation_work.groupby(
                [
                    "WELL",
                    "FORMATION_KEY",
                ],
                sort=False,
            )
        ):
            formation_profiles[
                (
                    well_name,
                    formation_key,
                )
            ] = create_prefix_profile(
                group_frame[
                    vertical_column
                ].to_numpy(
                    dtype=float
                ),
                group_frame[
                    "LITHOLOGY_ENCODED"
                ].to_numpy(
                    dtype=int
                ),
            )

    per_well_class_distribution = (
        pd.crosstab(
            work[
                "WELL"
            ],
            work[
                "LITHOLOGY_ENCODED"
            ],
        )
        .reindex(
            columns=CLASS_INDICES,
            fill_value=0,
        )
        .astype(float)
    )

    per_well_class_distribution = (
        per_well_class_distribution.div(
            per_well_class_distribution.sum(
                axis=1
            ),
            axis=0,
        )
    )

    global_well_balanced_prior = (
        per_well_class_distribution.mean(
            axis=0
        )
        .to_numpy(
            dtype=np.float64
        )
    )

    global_well_balanced_prior = (
        normalize_probability_rows(
            global_well_balanced_prior[
                None,
                :
            ]
        )[
            0
        ]
    )

    coordinate_matrix = (
        coordinate_table[
            [
                "X_LOC",
                "Y_LOC",
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    if len(coordinate_matrix) >= 2:
        pairwise = (
            coordinate_matrix[
                :,
                None,
                :
            ]
            - coordinate_matrix[
                None,
                :,
                :
            ]
        )

        distances = np.sqrt(
            np.sum(
                pairwise
                ** 2,
                axis=2,
            )
        )

        np.fill_diagonal(
            distances,
            np.inf,
        )

        nearest_distances = (
            distances.min(
                axis=1
            )
        )

        median_nearest_distance = float(
            np.median(
                nearest_distances[
                    np.isfinite(
                        nearest_distances
                    )
                ]
            )
        )

    else:
        median_nearest_distance = 1.0

    return {
        "vertical_column": vertical_column,
        "vertical_info": vertical_info,
        "coordinate_table": (
            coordinate_table
        ),
        "all_profiles": all_profiles,
        "formation_profiles": (
            formation_profiles
        ),
        "formation_available": (
            formation_available
        ),
        "global_well_balanced_prior": (
            global_well_balanced_prior
        ),
        "median_nearest_distance": max(
            median_nearest_distance,
            PROBABILITY_EPSILON,
        ),
        "training_wells": int(
            work[
                "WELL"
            ].nunique()
        ),
    }

# 8. Well-balanced XY + depth prior

In [13]:
def aggregate_neighbor_profiles(
    query_z,
    neighbor_wells,
    neighbor_weights,
    profile_lookup,
    half_window,
):
    query_z = np.asarray(
        query_z,
        dtype=np.float64,
    )

    n_rows = len(query_z)

    weighted_sum = np.zeros(
        (
            n_rows,
            len(CLASS_INDICES),
        ),
        dtype=np.float64,
    )

    total_weight = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    support_wells = np.zeros(
        n_rows,
        dtype=np.int16,
    )

    weighted_row_support = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    contribution_records = []

    for well_name, well_weight in zip(
        neighbor_wells,
        neighbor_weights,
    ):
        profile = profile_lookup.get(
            well_name
        )

        if profile is None:
            continue

        counts = counts_from_profile(
            profile,
            query_z,
            half_window,
        )

        totals = counts.sum(axis=1)

        supported = (
            totals
            >= MIN_ROWS_PER_WELL_WINDOW
        )

        if not supported.any():
            continue

        proportions = np.zeros_like(
            counts,
            dtype=np.float64,
        )

        proportions[
            supported
        ] = (
            counts[
                supported
            ]
            / totals[
                supported,
                None,
            ]
        )

        effective_weight = (
            float(well_weight)
            * supported.astype(
                np.float64
            )
        )

        weighted_sum += (
            proportions
            * effective_weight[
                :,
                None,
            ]
        )

        total_weight += effective_weight

        support_wells += (
            supported.astype(
                np.int16
            )
        )

        weighted_row_support += (
            totals
            * effective_weight
        )

        contribution_records.append({
            "proportions": proportions,
            "effective_weight": (
                effective_weight
            ),
        })

    aggregate = np.zeros_like(
        weighted_sum
    )

    valid = total_weight > 0

    aggregate[
        valid
    ] = (
        weighted_sum[
            valid
        ]
        / total_weight[
            valid,
            None,
        ]
    )

    average_rows = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    average_rows[
        valid
    ] = (
        weighted_row_support[
            valid
        ]
        / total_weight[
            valid
        ]
    )

    dispersion = np.ones(
        n_rows,
        dtype=np.float64,
    )

    if contribution_records:
        weighted_dispersion = np.zeros(
            n_rows,
            dtype=np.float64,
        )

        for record in (
            contribution_records
        ):
            total_variation = (
                0.5
                * np.abs(
                    record[
                        "proportions"
                    ]
                    - aggregate
                ).sum(axis=1)
            )

            weighted_dispersion += (
                total_variation
                * record[
                    "effective_weight"
                ]
            )

        dispersion[
            valid
        ] = (
            weighted_dispersion[
                valid
            ]
            / total_weight[
                valid
            ]
        )

    agreement = np.clip(
        1.0 - dispersion,
        0.0,
        1.0,
    )

    return {
        "aggregate": aggregate,
        "valid": valid,
        "support_wells": (
            support_wells
        ),
        "average_rows": (
            average_rows
        ),
        "agreement": agreement,
    }


def compute_geo_prior(
    raw_query,
    bank,
    k_neighbors,
    vertical_half_window,
):
    query = (
        raw_query.copy()
        .reset_index(drop=True)
    )

    vertical_column = (
        bank[
            "vertical_column"
        ]
    )

    required_query_columns = {
        "WELL",
        "X_LOC",
        "Y_LOC",
        vertical_column,
    }

    missing_query_columns = (
        required_query_columns
        - set(query.columns)
    )

    if missing_query_columns:
        raise KeyError(
            "Missing query geo-prior columns: "
            + str(
                sorted(
                    missing_query_columns
                )
            )
        )

    query[
        "X_LOC"
    ] = clean_numeric_series(
        query[
            "X_LOC"
        ]
    )

    query[
        "Y_LOC"
    ] = clean_numeric_series(
        query[
            "Y_LOC"
        ]
    )

    query[
        vertical_column
    ] = clean_numeric_series(
        query[
            vertical_column
        ]
    )

    n_rows = len(query)

    prior_matrix = np.tile(
        bank[
            "global_well_balanced_prior"
        ][
            None,
            :
        ],
        (
            n_rows,
            1,
        ),
    ).astype(
        np.float64
    )

    reliability_base = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    nearest_distance_output = np.full(
        n_rows,
        np.nan,
        dtype=np.float64,
    )

    support_wells_output = np.zeros(
        n_rows,
        dtype=np.int16,
    )

    average_rows_output = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    agreement_output = np.zeros(
        n_rows,
        dtype=np.float64,
    )

    formation_specific_output = np.zeros(
        n_rows,
        dtype=bool,
    )

    coordinate_table = (
        bank[
            "coordinate_table"
        ]
    )

    training_coordinates = (
        coordinate_table[
            [
                "X_LOC",
                "Y_LOC",
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    training_well_names = (
        coordinate_table[
            "WELL"
        ]
        .to_numpy()
    )

    if len(training_well_names) == 0:
        return {
            "prior": prior_matrix,
            "reliability_base": (
                reliability_base
            ),
            "nearest_distance": (
                nearest_distance_output
            ),
            "support_wells": (
                support_wells_output
            ),
            "average_rows": (
                average_rows_output
            ),
            "agreement": (
                agreement_output
            ),
            "formation_specific": (
                formation_specific_output
            ),
        }

    for query_well, row_indices in (
        query.groupby(
            "WELL",
            sort=False,
        ).groups.items()
    ):
        positions = np.asarray(
            list(row_indices),
            dtype=int,
        )

        well_x = float(
            query.loc[
                positions,
                "X_LOC",
            ].median()
        )

        well_y = float(
            query.loc[
                positions,
                "Y_LOC",
            ].median()
        )

        if (
            not np.isfinite(well_x)
            or not np.isfinite(well_y)
        ):
            continue

        differences = (
            training_coordinates
            - np.array([
                well_x,
                well_y,
            ])[
                None,
                :
            ]
        )

        distances = np.sqrt(
            np.sum(
                differences
                ** 2,
                axis=1,
            )
        )

        distances[
            training_well_names
            == query_well
        ] = np.inf

        finite_order = np.argsort(
            distances
        )

        finite_order = finite_order[
            np.isfinite(
                distances[
                    finite_order
                ]
            )
        ]

        if len(finite_order) == 0:
            continue

        selected_order = finite_order[
            :min(
                int(k_neighbors),
                len(finite_order),
            )
        ]

        neighbor_wells = (
            training_well_names[
                selected_order
            ]
        )

        neighbor_distances = (
            distances[
                selected_order
            ]
        )

        adaptive_bandwidth = max(
            float(
                neighbor_distances[
                    -1
                ]
            ),
            bank[
                "median_nearest_distance"
            ],
            PROBABILITY_EPSILON,
        )

        neighbor_weights = np.exp(
            -0.5
            * (
                neighbor_distances
                / adaptive_bandwidth
            )
            ** 2
        )

        nearest_distance = float(
            neighbor_distances[
                0
            ]
        )

        distance_factor = float(
            np.exp(
                -0.5
                * (
                    nearest_distance
                    / bank[
                        "median_nearest_distance"
                    ]
                )
                ** 2
            )
        )

        query_z_all = query.loc[
            positions,
            vertical_column,
        ].to_numpy(
            dtype=float
        )

        valid_z_mask = np.isfinite(
            query_z_all
        )

        if not valid_z_mask.any():
            continue

        valid_positions = positions[
            valid_z_mask
        ]

        valid_query_z = query_z_all[
            valid_z_mask
        ]

        if (
            bank[
                "formation_available"
            ]
            and "FORMATION" in (
                query.columns
            )
        ):
            formation_values = (
                query.loc[
                    valid_positions,
                    "FORMATION",
                ]
            )

            formation_keys = np.where(
                formation_values.notna(),
                formation_values.astype(
                    str
                ),
                "__MISSING__",
            )

        else:
            formation_keys = np.full(
                len(valid_positions),
                "__NO_FORMATION__",
                dtype=object,
            )

        for formation_key in pd.unique(
            formation_keys
        ):
            local_selector = (
                formation_keys
                == formation_key
            )

            local_positions = (
                valid_positions[
                    local_selector
                ]
            )

            local_query_z = (
                valid_query_z[
                    local_selector
                ]
            )

            selected_result = None
            formation_rows_used = (
                np.zeros(
                    len(local_positions),
                    dtype=bool,
                )
            )

            use_formation = bool(
                bank[
                    "formation_available"
                ]
                and formation_key
                not in {
                    "__MISSING__",
                    "__NO_FORMATION__",
                }
            )

            if use_formation:
                formation_lookup = {
                    well_name: bank[
                        "formation_profiles"
                    ].get(
                        (
                            well_name,
                            str(
                                formation_key
                            ),
                        )
                    )
                    for well_name in (
                        neighbor_wells
                    )
                }

                formation_result = (
                    aggregate_neighbor_profiles(
                        local_query_z,
                        neighbor_wells,
                        neighbor_weights,
                        formation_lookup,
                        vertical_half_window,
                    )
                )

                formation_usable = (
                    formation_result[
                        "support_wells"
                    ]
                    >= MIN_FORMATION_SUPPORT_WELLS
                )

                if formation_usable.any():
                    all_result = (
                        aggregate_neighbor_profiles(
                            local_query_z,
                            neighbor_wells,
                            neighbor_weights,
                            bank[
                                "all_profiles"
                            ],
                            vertical_half_window,
                        )
                    )

                    for key in [
                        "aggregate",
                        "valid",
                        "support_wells",
                        "average_rows",
                        "agreement",
                    ]:
                        all_result[
                            key
                        ][
                            formation_usable
                        ] = formation_result[
                            key
                        ][
                            formation_usable
                        ]

                    selected_result = (
                        all_result
                    )

                    formation_rows_used = (
                        formation_usable
                    )

            if selected_result is None:
                selected_result = (
                    aggregate_neighbor_profiles(
                        local_query_z,
                        neighbor_wells,
                        neighbor_weights,
                        bank[
                            "all_profiles"
                        ],
                        vertical_half_window,
                    )
                )

            valid_local = selected_result[
                "valid"
            ]

            if not valid_local.any():
                continue

            local_prior = selected_result[
                "aggregate"
            ]

            support_wells = selected_result[
                "support_wells"
            ].astype(
                np.float64
            )

            smoothing = float(
                PRIOR_SMOOTHING_EQUIVALENT_WELLS
            )

            smoothed_prior = (
                support_wells[
                    :,
                    None,
                ]
                * local_prior
                + smoothing
                * bank[
                    "global_well_balanced_prior"
                ][
                    None,
                    :
                ]
            ) / (
                support_wells[
                    :,
                    None,
                ]
                + smoothing
            )

            smoothed_prior = (
                normalize_probability_rows(
                    smoothed_prior
                )
            )

            support_factor = np.clip(
                support_wells
                / float(
                    MIN_WELLS_FULL_TRUST
                ),
                0.0,
                1.0,
            )

            depth_factor = np.clip(
                selected_result[
                    "average_rows"
                ]
                / float(
                    ROWS_PER_WELL_FULL_TRUST
                ),
                0.0,
                1.0,
            )

            agreement_factor = np.clip(
                selected_result[
                    "agreement"
                ],
                0.0,
                1.0,
            )

            local_reliability = (
                support_factor
                * depth_factor
                * agreement_factor
                * distance_factor
            )

            prior_matrix[
                local_positions
            ] = smoothed_prior

            reliability_base[
                local_positions
            ] = local_reliability

            nearest_distance_output[
                local_positions
            ] = nearest_distance

            support_wells_output[
                local_positions
            ] = selected_result[
                "support_wells"
            ]

            average_rows_output[
                local_positions
            ] = selected_result[
                "average_rows"
            ]

            agreement_output[
                local_positions
            ] = selected_result[
                "agreement"
            ]

            formation_specific_output[
                local_positions
            ] = formation_rows_used

    return {
        "prior": prior_matrix,
        "reliability_base": (
            reliability_base
        ),
        "nearest_distance": (
            nearest_distance_output
        ),
        "support_wells": (
            support_wells_output
        ),
        "average_rows": (
            average_rows_output
        ),
        "agreement": (
            agreement_output
        ),
        "formation_specific": (
            formation_specific_output
        ),
    }

# 9. Adaptive bounded fusion

In [14]:
def fuse_evidence_and_geo_prior(
    evidence_probabilities,
    raw_training_prior,
    resampled_training_prior,
    geo_prior,
    geo_global_prior,
    reliability_base,
    lambda_max,
    spatial_ratio_clip,
    resampling_prior_correction_power,
):
    corrected_evidence = (
        apply_resampling_prior_correction(
            evidence_probabilities,
            raw_training_prior,
            resampled_training_prior,
            power=(
                resampling_prior_correction_power
            ),
        )
    )

    global_prior = np.clip(
        geo_global_prior,
        PROBABILITY_EPSILON,
        1.0,
    )

    local_prior = np.clip(
        geo_prior,
        PROBABILITY_EPSILON,
        1.0,
    )

    log_spatial_ratio = (
        np.log(local_prior)
        - np.log(
            global_prior[
                None,
                :
            ]
        )
    )

    maximum_log_ratio = float(
        np.log(
            spatial_ratio_clip
        )
    )

    log_spatial_ratio = np.clip(
        log_spatial_ratio,
        -maximum_log_ratio,
        maximum_log_ratio,
    )

    row_lambda = (
        float(lambda_max)
        * np.asarray(
            reliability_base,
            dtype=np.float64,
        )
    )

    log_final_scores = (
        np.log(
            np.clip(
                corrected_evidence,
                PROBABILITY_EPSILON,
                1.0,
            )
        )
        + row_lambda[
            :,
            None,
        ]
        * log_spatial_ratio
    )

    final_probabilities = (
        softmax_from_log_scores(
            log_final_scores
        )
    )

    return {
        "corrected_evidence_probabilities": (
            corrected_evidence
        ),
        "final_probabilities": (
            final_probabilities
        ),
        "row_lambda": row_lambda,
        "log_spatial_ratio": (
            log_spatial_ratio
        ),
    }

In [15]:
with open(
    LEGACY_REPRODUCTION_GATE_PATH,
    "r",
    encoding="utf-8",
) as file:
    legacy_reproduction_gate = (
        json.load(file)
    )

if not legacy_reproduction_gate.get(
    "overall_reproduction_passed",
    False,
):
    raise RuntimeError(
        "Exact Legacy reproduction gate did not pass."
    )

train_raw = load_csv_full(
    TRAIN_PATH
)

if "WELL" not in train_raw.columns:
    raise ValueError(
        "WELL column is missing from train.csv"
    )

train_raw[
    "ORIGINAL_ROW_ID"
] = np.arange(
    len(train_raw),
    dtype=np.int64,
)

fold_assignment_raw = pd.read_csv(
    FOLD_ASSIGNMENT_PATH
)

well_candidates = [
    column
    for column in (
        fold_assignment_raw.columns
    )
    if column.upper() == "WELL"
]

fold_candidates = [
    column
    for column in (
        fold_assignment_raw.columns
    )
    if (
        "FOLD"
        in column.upper()
        and "SCORE"
        not in column.upper()
    )
]

if (
    not well_candidates
    or not fold_candidates
):
    raise ValueError(
        "Could not infer WELL/FOLD columns "
        "from fold assignment file."
    )

fold_assignment = (
    fold_assignment_raw[
        [
            well_candidates[0],
            fold_candidates[0],
        ]
    ]
    .rename(
        columns={
            well_candidates[0]: "WELL",
            fold_candidates[0]: (
                "GROUP_FOLD"
            ),
        }
    )
    .drop_duplicates(
        subset=["WELL"]
    )
)

fold_assignment[
    "GROUP_FOLD"
] = pd.to_numeric(
    fold_assignment[
        "GROUP_FOLD"
    ],
    errors="raise",
).astype(int)

train_raw[
    "GROUP_FOLD"
] = train_raw[
    "WELL"
].map(
    dict(
        zip(
            fold_assignment[
                "WELL"
            ],
            fold_assignment[
                "GROUP_FOLD"
            ],
        )
    )
)

if train_raw[
    "GROUP_FOLD"
].isna().any():
    missing_wells = sorted(
        train_raw.loc[
            train_raw[
                "GROUP_FOLD"
            ].isna(),
            "WELL",
        ]
        .dropna()
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "Wells without grouped fold assignment: "
        + str(
            missing_wells[:20]
        )
    )

train_raw[
    "GROUP_FOLD"
] = train_raw[
    "GROUP_FOLD"
].astype(int)

train_raw[
    TARGET_COLUMN
] = pd.to_numeric(
    train_raw[
        TARGET_COLUMN
    ],
    errors="raise",
).astype(int)

print("Train:", train_raw.shape)
print(
    "Wells:",
    train_raw["WELL"].nunique(),
)
print(
    "Folds:",
    sorted(
        train_raw[
            "GROUP_FOLD"
        ].unique()
    ),
)

Loading train.csv with {'sep': ',', 'encoding': 'utf-8'}
Train: (1170511, 31)
Wells: 98
Folds: [1, 2, 3, 4, 5]


# 11. Grouped CV evidence cache

In [16]:
def evidence_cache_paths(
    outer_fold,
):
    prefix = (
        f"outer_fold_{outer_fold}"
        "__legacy_causal_evidence"
    )

    return {
        "predictions": (
            SHARED_CACHE_DIR
            / f"{prefix}.parquet"
        ),
        "metadata": (
            SHARED_CACHE_DIR
            / f"{prefix}.json"
        ),
        "resampling": (
            SHARED_CACHE_DIR
            / f"{prefix}__resampling.csv"
        ),
    }


def build_or_load_fold_evidence(
    outer_fold,
):
    paths = evidence_cache_paths(
        outer_fold
    )

    if (
        paths[
            "predictions"
        ].exists()
        and paths[
            "metadata"
        ].exists()
        and paths[
            "resampling"
        ].exists()
    ):
        predictions = pd.read_parquet(
            paths[
                "predictions"
            ]
        )

        with open(
            paths[
                "metadata"
            ],
            "r",
            encoding="utf-8",
        ) as file:
            metadata = json.load(file)

        if metadata.get(
            "fold_gate_passed",
            False,
        ):
            print(
                "Loading cached evidence fold:",
                outer_fold,
            )

            return {
                "predictions": predictions,
                "metadata": metadata,
                "resampling_audit": (
                    pd.read_csv(
                        paths[
                            "resampling"
                        ]
                    )
                ),
            }

    print(
        "\nBuilding evidence fold:",
        outer_fold,
    )

    validation_mask = (
        train_raw[
            "GROUP_FOLD"
        ].to_numpy()
        == outer_fold
    )

    raw_outer_train = (
        train_raw.loc[
            ~validation_mask
        ]
        .drop(
            columns=[
                "GROUP_FOLD"
            ]
        )
        .copy()
        .reset_index(drop=True)
    )

    raw_outer_validation = (
        train_raw.loc[
            validation_mask
        ]
        .drop(
            columns=[
                "GROUP_FOLD"
            ]
        )
        .copy()
        .reset_index(drop=True)
    )

    train_wells = set(
        raw_outer_train[
            "WELL"
        ].dropna()
    )

    validation_wells = set(
        raw_outer_validation[
            "WELL"
        ].dropna()
    )

    if train_wells & validation_wells:
        raise RuntimeError(
            "WELL leakage in outer fold."
        )

    prepared = (
        prepare_evidence_train_and_evaluation(
            raw_outer_train,
            raw_outer_validation,
        )
    )

    fitted = fit_evidence_RF(
        prepared
    )

    evidence_probabilities = (
        fitted[
            "probabilities"
        ]
    )

    predictions = (
        prepared[
            "evaluation_metadata"
        ]
        .copy()
    )

    predictions[
        "outer_fold"
    ] = int(outer_fold)

    for class_index, class_code in enumerate(
        CLASS_CODES
    ):
        predictions[
            f"evidence_prob_{class_code}"
        ] = (
            evidence_probabilities[
                :,
                class_index,
            ].astype(
                np.float32
            )
        )

    probability_matrix = predictions[
        [
            f"evidence_prob_{code}"
            for code in CLASS_CODES
        ]
    ].to_numpy(
        dtype=np.float64
    )

    metadata = {
        "outer_fold": int(outer_fold),
        "train_rows_raw": int(
            prepared[
                "raw_training_rows"
            ]
        ),
        "train_rows_resampled": int(
            prepared[
                "resampled_training_rows"
            ]
        ),
        "validation_rows": int(
            len(predictions)
        ),
        "train_wells": int(
            len(train_wells)
        ),
        "validation_wells": int(
            len(validation_wells)
        ),
        "well_overlap_count": 0,
        "fit_seconds": float(
            fitted[
                "fit_seconds"
            ]
        ),
        "raw_training_prior": (
            prepared[
                "raw_training_prior"
            ].tolist()
        ),
        "resampled_training_prior": (
            prepared[
                "resampled_training_prior"
            ].tolist()
        ),
        "evidence_feature_count": int(
            len(
                LEGACY_EVIDENCE_FEATURES
            )
        ),
        "X_LOC_removed_from_evidence": True,
        "Y_LOC_removed_from_evidence": True,
        "causal_engineered_features_preserved": True,
        "probabilities_finite": bool(
            np.isfinite(
                probability_matrix
            ).all()
        ),
        "probabilities_sum_to_one": bool(
            np.allclose(
                probability_matrix.sum(
                    axis=1
                ),
                1.0,
                rtol=1e-6,
                atol=1e-6,
            )
        ),
        "official_test_unused": True,
    }

    metadata[
        "fold_gate_passed"
    ] = bool(
        metadata[
            "well_overlap_count"
        ]
        == 0
        and metadata[
            "X_LOC_removed_from_evidence"
        ]
        and metadata[
            "Y_LOC_removed_from_evidence"
        ]
        and metadata[
            "causal_engineered_features_preserved"
        ]
        and metadata[
            "probabilities_finite"
        ]
        and metadata[
            "probabilities_sum_to_one"
        ]
        and metadata[
            "official_test_unused"
        ]
    )

    if not metadata[
        "fold_gate_passed"
    ]:
        raise RuntimeError(
            "Evidence fold gate failed."
        )

    predictions.to_parquet(
        paths[
            "predictions"
        ],
        index=False,
    )

    prepared[
        "resampling_audit"
    ].to_csv(
        paths[
            "resampling"
        ],
        index=False,
    )

    with open(
        paths[
            "metadata"
        ],
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(metadata),
            file,
            ensure_ascii=False,
            indent=2,
        )

    del fitted
    del prepared
    gc.collect()

    return {
        "predictions": predictions,
        "metadata": metadata,
        "resampling_audit": (
            pd.read_csv(
                paths[
                    "resampling"
                ]
            )
        ),
    }

In [17]:
if RUN_MODE in {
    "pilot",
    "full_cv",
}:
    candidate_fold_metric_rows = []
    candidate_well_metric_frames = []
    fold_prior_diagnostic_rows = []
    evidence_fold_metadata_rows = []

    for outer_fold in (
        ACTIVE_OUTER_FOLDS
    ):
        print(
            "\n"
            + "=" * 110
        )

        print(
            "OUTER FOLD",
            outer_fold,
        )

        fold_evidence = (
            build_or_load_fold_evidence(
                outer_fold
            )
        )

        evidence_predictions = (
            fold_evidence[
                "predictions"
            ]
            .copy()
            .reset_index(drop=True)
        )

        evidence_fold_metadata_rows.append(
            fold_evidence[
                "metadata"
            ]
        )

        raw_outer_train = (
            train_raw.loc[
                train_raw[
                    "GROUP_FOLD"
                ]
                != outer_fold
            ]
            .drop(
                columns=[
                    "GROUP_FOLD"
                ]
            )
            .copy()
            .reset_index(drop=True)
        )

        raw_outer_validation = (
            train_raw.loc[
                train_raw[
                    "GROUP_FOLD"
                ]
                == outer_fold
            ]
            .drop(
                columns=[
                    "GROUP_FOLD"
                ]
            )
            .copy()
            .reset_index(drop=True)
        )

        if not np.array_equal(
            evidence_predictions[
                "ORIGINAL_ROW_ID"
            ].to_numpy(
                dtype=np.int64
            ),
            raw_outer_validation[
                "ORIGINAL_ROW_ID"
            ].to_numpy(
                dtype=np.int64
            ),
        ):
            raise RuntimeError(
                "Evidence/query row order mismatch."
            )

        evidence_probabilities = (
            evidence_predictions[
                [
                    f"evidence_prob_{code}"
                    for code in CLASS_CODES
                ]
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        y_validation = (
            evidence_predictions[
                "LITHOLOGY_ENCODED"
            ]
            .to_numpy(
                dtype=int
            )
        )

        bank = build_geo_prior_bank(
            raw_outer_train
        )

        raw_training_prior = np.asarray(
            fold_evidence[
                "metadata"
            ][
                "raw_training_prior"
            ],
            dtype=np.float64,
        )

        resampled_training_prior = np.asarray(
            fold_evidence[
                "metadata"
            ][
                "resampled_training_prior"
            ],
            dtype=np.float64,
        )

        for (
            k_neighbors,
            vertical_half_window,
        ) in product(
            K_NEIGHBORS_GRID,
            VERTICAL_HALF_WINDOW_GRID,
        ):
            print(
                "Geo prior:",
                "K=",
                k_neighbors,
                "| half-window=",
                vertical_half_window,
            )

            geo_result = compute_geo_prior(
                raw_outer_validation,
                bank,
                k_neighbors=k_neighbors,
                vertical_half_window=(
                    vertical_half_window
                ),
            )

            fold_prior_diagnostic_rows.append({
                "outer_fold": int(
                    outer_fold
                ),
                "k_neighbors": int(
                    k_neighbors
                ),
                "vertical_half_window": float(
                    vertical_half_window
                ),
                "vertical_column": (
                    bank[
                        "vertical_column"
                    ]
                ),
                "mean_reliability_base": float(
                    geo_result[
                        "reliability_base"
                    ].mean()
                ),
                "fraction_zero_reliability": float(
                    (
                        geo_result[
                            "reliability_base"
                        ]
                        <= 0
                    ).mean()
                ),
                "median_nearest_distance": float(
                    np.nanmedian(
                        geo_result[
                            "nearest_distance"
                        ]
                    )
                ),
                "mean_support_wells": float(
                    geo_result[
                        "support_wells"
                    ].mean()
                ),
                "mean_depth_rows_per_well": float(
                    geo_result[
                        "average_rows"
                    ].mean()
                ),
                "mean_agreement": float(
                    geo_result[
                        "agreement"
                    ].mean()
                ),
                "formation_specific_fraction": float(
                    geo_result[
                        "formation_specific"
                    ].mean()
                ),
            })

            matching_candidates = (
                candidate_registry.loc[
                    (
                        candidate_registry[
                            "k_neighbors"
                        ]
                        == k_neighbors
                    )
                    & (
                        candidate_registry[
                            "vertical_half_window"
                        ]
                        == vertical_half_window
                    )
                ]
            )

            for candidate in (
                matching_candidates.itertuples(
                    index=False
                )
            ):
                fusion = (
                    fuse_evidence_and_geo_prior(
                        evidence_probabilities=(
                            evidence_probabilities
                        ),
                        raw_training_prior=(
                            raw_training_prior
                        ),
                        resampled_training_prior=(
                            resampled_training_prior
                        ),
                        geo_prior=(
                            geo_result[
                                "prior"
                            ]
                        ),
                        geo_global_prior=(
                            bank[
                                "global_well_balanced_prior"
                            ]
                        ),
                        reliability_base=(
                            geo_result[
                                "reliability_base"
                            ]
                        ),
                        lambda_max=(
                            candidate.lambda_max
                        ),
                        spatial_ratio_clip=(
                            candidate.spatial_ratio_clip
                        ),
                        resampling_prior_correction_power=(
                            candidate.resampling_prior_correction_power
                        ),
                    )
                )

                final_probabilities = (
                    fusion[
                        "final_probabilities"
                    ]
                )

                (
                    raw_predictions,
                    refined_predictions,
                    refinement_mask,
                ) = apply_legacy_chalk_refinement(
                    final_probabilities
                )

                raw_metrics = (
                    calculate_metrics(
                        y_validation,
                        raw_predictions,
                        probabilities=(
                            final_probabilities
                        ),
                    )
                )

                refined_metrics = (
                    calculate_metrics(
                        y_validation,
                        refined_predictions,
                    )
                )

                per_well = (
                    calculate_per_well_metrics(
                        evidence_predictions,
                        final_probabilities,
                        raw_predictions,
                        refined_predictions,
                    )
                )

                per_well[
                    "outer_fold"
                ] = int(
                    outer_fold
                )

                per_well[
                    "candidate_id"
                ] = (
                    candidate.candidate_id
                )

                candidate_well_metric_frames.append(
                    per_well
                )

                candidate_fold_metric_rows.append({
                    "outer_fold": int(
                        outer_fold
                    ),
                    "candidate_id": (
                        candidate.candidate_id
                    ),
                    "k_neighbors": int(
                        candidate.k_neighbors
                    ),
                    "vertical_half_window": float(
                        candidate.vertical_half_window
                    ),
                    "lambda_max": float(
                        candidate.lambda_max
                    ),
                    "spatial_ratio_clip": float(
                        candidate.spatial_ratio_clip
                    ),
                    "resampling_prior_correction_power": float(
                        candidate.resampling_prior_correction_power
                    ),
                    "raw_accuracy": (
                        raw_metrics[
                            "accuracy"
                        ]
                    ),
                    "raw_balanced_accuracy": (
                        raw_metrics[
                            "balanced_accuracy"
                        ]
                    ),
                    "raw_macro_f1_present_classes": (
                        raw_metrics[
                            "macro_f1_present_classes"
                        ]
                    ),
                    "raw_macro_f1_all_12_classes": (
                        raw_metrics[
                            "macro_f1_all_12_classes"
                        ]
                    ),
                    "raw_log_loss": (
                        raw_metrics[
                            "log_loss"
                        ]
                    ),
                    "refined_accuracy": (
                        refined_metrics[
                            "accuracy"
                        ]
                    ),
                    "refined_balanced_accuracy": (
                        refined_metrics[
                            "balanced_accuracy"
                        ]
                    ),
                    "refined_macro_f1_present_classes": (
                        refined_metrics[
                            "macro_f1_present_classes"
                        ]
                    ),
                    "refined_macro_f1_all_12_classes": (
                        refined_metrics[
                            "macro_f1_all_12_classes"
                        ]
                    ),
                    "refinement_changed_fraction": float(
                        refinement_mask.mean()
                    ),
                    "mean_row_lambda": float(
                        fusion[
                            "row_lambda"
                        ].mean()
                    ),
                    "fraction_zero_lambda": float(
                        (
                            fusion[
                                "row_lambda"
                            ]
                            <= 0
                        ).mean()
                    ),
                })

            del geo_result
            gc.collect()

        del bank
        del evidence_predictions
        del evidence_probabilities
        gc.collect()

    candidate_fold_metrics = pd.DataFrame(
        candidate_fold_metric_rows
    )

    candidate_per_well_metrics = pd.concat(
        candidate_well_metric_frames,
        ignore_index=True,
    )

    prior_diagnostics = pd.DataFrame(
        fold_prior_diagnostic_rows
    )

    evidence_fold_metadata = pd.DataFrame(
        evidence_fold_metadata_rows
    )

    candidate_summary = (
        candidate_per_well_metrics.groupby(
            "candidate_id",
            as_index=False,
        )
        .agg(
            wells=(
                "WELL",
                "nunique",
            ),
            well_balanced_refined_macro_f1=(
                "refined_macro_f1_present_classes",
                "mean",
            ),
            well_balanced_refined_balanced_accuracy=(
                "refined_balanced_accuracy",
                "mean",
            ),
            well_balanced_refined_accuracy=(
                "refined_accuracy",
                "mean",
            ),
            well_balanced_raw_macro_f1=(
                "raw_macro_f1_present_classes",
                "mean",
            ),
            well_balanced_log_loss=(
                "log_loss",
                "mean",
            ),
            worst_well_log_loss=(
                "log_loss",
                "max",
            ),
        )
        .merge(
            candidate_registry,
            on="candidate_id",
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            [
                "well_balanced_refined_macro_f1",
                "well_balanced_refined_balanced_accuracy",
                "well_balanced_log_loss",
                "lambda_max",
                "candidate_id",
            ],
            ascending=[
                False,
                False,
                True,
                True,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    selected_candidate = (
        candidate_summary.iloc[
            0
        ].to_dict()
    )

    print(
        "\nSelected fusion candidate:"
    )

    display(
        pd.DataFrame([
            selected_candidate
        ])
    )

    candidate_fold_metrics.to_csv(
        OUTPUT_DIR
        / "01_candidate_fold_metrics.csv",
        index=False,
    )

    candidate_per_well_metrics.to_csv(
        OUTPUT_DIR
        / "02_candidate_per_well_metrics.csv",
        index=False,
    )

    candidate_summary.to_csv(
        OUTPUT_DIR
        / "03_candidate_well_balanced_summary.csv",
        index=False,
    )

    prior_diagnostics.to_csv(
        OUTPUT_DIR
        / "04_geo_prior_diagnostics.csv",
        index=False,
    )

    evidence_fold_metadata.to_csv(
        OUTPUT_DIR
        / "05_evidence_fold_audit.csv",
        index=False,
    )


OUTER FOLD 1
Loading cached evidence fold: 1
Geo prior: K= 3 | half-window= 15.0
Geo prior: K= 3 | half-window= 30.0
Geo prior: K= 3 | half-window= 60.0
Geo prior: K= 5 | half-window= 15.0
Geo prior: K= 5 | half-window= 30.0
Geo prior: K= 5 | half-window= 60.0

OUTER FOLD 2

Building evidence fold: 2


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   16.7s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:  3.7min
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:  6.6min
[Parallel(n_jobs=-1)]: Done 800 out of 800 | elapsed:  6.8min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.2s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.7s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.6s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    2.9s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    3.0s finished


Geo prior: K= 3 | half-window= 15.0
Geo prior: K= 3 | half-window= 30.0
Geo prior: K= 3 | half-window= 60.0
Geo prior: K= 5 | half-window= 15.0
Geo prior: K= 5 | half-window= 30.0
Geo prior: K= 5 | half-window= 60.0

OUTER FOLD 3

Building evidence fold: 3


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   16.1s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:  3.6min
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:  6.9min
[Parallel(n_jobs=-1)]: Done 800 out of 800 | elapsed:  7.1min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.2s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.8s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.9s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    3.4s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    3.5s finished


Geo prior: K= 3 | half-window= 15.0
Geo prior: K= 3 | half-window= 30.0
Geo prior: K= 3 | half-window= 60.0
Geo prior: K= 5 | half-window= 15.0
Geo prior: K= 5 | half-window= 30.0
Geo prior: K= 5 | half-window= 60.0

OUTER FOLD 4

Building evidence fold: 4


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   17.4s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:  4.1min
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:  7.6min
[Parallel(n_jobs=-1)]: Done 800 out of 800 | elapsed:  7.8min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.2s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.8s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.7s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    3.4s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    3.4s finished


Geo prior: K= 3 | half-window= 15.0
Geo prior: K= 3 | half-window= 30.0
Geo prior: K= 3 | half-window= 60.0
Geo prior: K= 5 | half-window= 15.0
Geo prior: K= 5 | half-window= 30.0
Geo prior: K= 5 | half-window= 60.0

OUTER FOLD 5

Building evidence fold: 5


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   15.3s
[Parallel(n_jobs=-1)]: Done 180 tasks      | elapsed:  1.7min
[Parallel(n_jobs=-1)]: Done 430 tasks      | elapsed:  3.9min
[Parallel(n_jobs=-1)]: Done 780 tasks      | elapsed:  7.1min
[Parallel(n_jobs=-1)]: Done 800 out of 800 | elapsed:  7.3min finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.2s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.8s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    2.0s
[Parallel(n_jobs=10)]: Done 780 tasks      | elapsed:    3.5s
[Parallel(n_jobs=10)]: Done 800 out of 800 | elapsed:    3.5s finished


Geo prior: K= 3 | half-window= 15.0
Geo prior: K= 3 | half-window= 30.0
Geo prior: K= 3 | half-window= 60.0
Geo prior: K= 5 | half-window= 15.0
Geo prior: K= 5 | half-window= 30.0
Geo prior: K= 5 | half-window= 60.0

Selected fusion candidate:


,candidate_id,wells,well_balanced_refined_macro_f1,well_balanced_refined_balanced_accuracy,well_balanced_refined_accuracy,well_balanced_raw_macro_f1,well_balanced_log_loss,worst_well_log_loss,k_neighbors,vertical_half_window,lambda_max,spatial_ratio_clip,resampling_prior_correction_power,formation_hierarchy,well_balanced_prior,adaptive_support_weight,adaptive_distance_weight,adaptive_depth_weight,adaptive_agreement_weight,global_fallback
0,k3__z60__lam0.75__clip2__priorcorr0,98,0.38569,0.397469,0.714595,0.386646,0.838938,2.00673,3,60.0,0.75,2.0,0.0,True,True,True,True,True,True,True


In [18]:
if RUN_MODE in {
    "pilot",
    "full_cv",
}:
    selected_oof_parts = []
    selected_fold_metric_rows = []
    selected_per_class_frames = []
    selected_per_well_frames = []

    selected_k = int(
        selected_candidate[
            "k_neighbors"
        ]
    )

    selected_window = float(
        selected_candidate[
            "vertical_half_window"
        ]
    )

    selected_lambda_max = float(
        selected_candidate[
            "lambda_max"
        ]
    )

    selected_clip = float(
        selected_candidate[
            "spatial_ratio_clip"
        ]
    )

    selected_prior_power = float(
        selected_candidate[
            "resampling_prior_correction_power"
        ]
    )

    for outer_fold in (
        ACTIVE_OUTER_FOLDS
    ):
        fold_evidence = (
            build_or_load_fold_evidence(
                outer_fold
            )
        )

        evidence_predictions = (
            fold_evidence[
                "predictions"
            ]
            .copy()
            .reset_index(drop=True)
        )

        raw_outer_train = (
            train_raw.loc[
                train_raw[
                    "GROUP_FOLD"
                ]
                != outer_fold
            ]
            .drop(
                columns=[
                    "GROUP_FOLD"
                ]
            )
            .copy()
            .reset_index(drop=True)
        )

        raw_outer_validation = (
            train_raw.loc[
                train_raw[
                    "GROUP_FOLD"
                ]
                == outer_fold
            ]
            .drop(
                columns=[
                    "GROUP_FOLD"
                ]
            )
            .copy()
            .reset_index(drop=True)
        )

        bank = build_geo_prior_bank(
            raw_outer_train
        )

        geo_result = compute_geo_prior(
            raw_outer_validation,
            bank,
            k_neighbors=selected_k,
            vertical_half_window=(
                selected_window
            ),
        )

        evidence_probabilities = (
            evidence_predictions[
                [
                    f"evidence_prob_{code}"
                    for code in CLASS_CODES
                ]
            ]
            .to_numpy(
                dtype=np.float64
            )
        )

        fusion = (
            fuse_evidence_and_geo_prior(
                evidence_probabilities=(
                    evidence_probabilities
                ),
                raw_training_prior=np.asarray(
                    fold_evidence[
                        "metadata"
                    ][
                        "raw_training_prior"
                    ],
                    dtype=np.float64,
                ),
                resampled_training_prior=np.asarray(
                    fold_evidence[
                        "metadata"
                    ][
                        "resampled_training_prior"
                    ],
                    dtype=np.float64,
                ),
                geo_prior=(
                    geo_result[
                        "prior"
                    ]
                ),
                geo_global_prior=(
                    bank[
                        "global_well_balanced_prior"
                    ]
                ),
                reliability_base=(
                    geo_result[
                        "reliability_base"
                    ]
                ),
                lambda_max=(
                    selected_lambda_max
                ),
                spatial_ratio_clip=(
                    selected_clip
                ),
                resampling_prior_correction_power=(
                    selected_prior_power
                ),
            )
        )

        final_probabilities = (
            fusion[
                "final_probabilities"
            ]
        )

        (
            raw_predictions,
            refined_predictions,
            refinement_mask,
        ) = apply_legacy_chalk_refinement(
            final_probabilities
        )

        y_true = (
            evidence_predictions[
                "LITHOLOGY_ENCODED"
            ]
            .to_numpy(
                dtype=int
            )
        )

        raw_metrics = calculate_metrics(
            y_true,
            raw_predictions,
            probabilities=(
                final_probabilities
            ),
        )

        refined_metrics = (
            calculate_metrics(
                y_true,
                refined_predictions,
            )
        )

        selected_fold_metric_rows.append({
            "outer_fold": int(
                outer_fold
            ),
            **{
                f"raw_{key}": value
                for key, value in (
                    raw_metrics.items()
                )
            },
            **{
                f"refined_{key}": value
                for key, value in (
                    refined_metrics.items()
                )
            },
            "mean_lambda": float(
                fusion[
                    "row_lambda"
                ].mean()
            ),
            "zero_lambda_fraction": float(
                (
                    fusion[
                        "row_lambda"
                    ]
                    <= 0
                ).mean()
            ),
        })

        per_class_raw = (
            calculate_per_class_metrics(
                y_true,
                raw_predictions,
            )
        )

        per_class_raw[
            "prediction_variant"
        ] = "raw_fused"

        per_class_refined = (
            calculate_per_class_metrics(
                y_true,
                refined_predictions,
            )
        )

        per_class_refined[
            "prediction_variant"
        ] = (
            "legacy_refined_fused"
        )

        per_class = pd.concat(
            [
                per_class_raw,
                per_class_refined,
            ],
            ignore_index=True,
        )

        per_class[
            "outer_fold"
        ] = int(outer_fold)

        selected_per_class_frames.append(
            per_class
        )

        per_well = (
            calculate_per_well_metrics(
                evidence_predictions,
                final_probabilities,
                raw_predictions,
                refined_predictions,
            )
        )

        per_well[
            "outer_fold"
        ] = int(outer_fold)

        selected_per_well_frames.append(
            per_well
        )

        selected_oof = (
            evidence_predictions.copy()
        )

        selected_oof[
            "raw_fused_prediction_index"
        ] = raw_predictions

        selected_oof[
            "legacy_refined_fused_prediction_index"
        ] = refined_predictions

        selected_oof[
            "legacy_refinement_modified"
        ] = refinement_mask

        selected_oof[
            "adaptive_lambda"
        ] = fusion[
            "row_lambda"
        ]

        selected_oof[
            "geo_nearest_distance"
        ] = geo_result[
            "nearest_distance"
        ]

        selected_oof[
            "geo_support_wells"
        ] = geo_result[
            "support_wells"
        ]

        selected_oof[
            "geo_average_rows_per_well"
        ] = geo_result[
            "average_rows"
        ]

        selected_oof[
            "geo_agreement"
        ] = geo_result[
            "agreement"
        ]

        selected_oof[
            "geo_formation_specific"
        ] = geo_result[
            "formation_specific"
        ]

        for class_index, class_code in enumerate(
            CLASS_CODES
        ):
            selected_oof[
                f"final_prob_{class_code}"
            ] = (
                final_probabilities[
                    :,
                    class_index,
                ].astype(
                    np.float32
                )
            )

            selected_oof[
                f"geo_prior_{class_code}"
            ] = (
                geo_result[
                    "prior"
                ][
                    :,
                    class_index,
                ].astype(
                    np.float32
                )
            )

        selected_oof_parts.append(
            selected_oof
        )

        del bank
        del geo_result
        del fusion
        del final_probabilities
        gc.collect()

    selected_oof_predictions = (
        pd.concat(
            selected_oof_parts,
            ignore_index=True,
        )
        .sort_values(
            "ORIGINAL_ROW_ID"
        )
        .reset_index(drop=True)
    )

    selected_fold_metrics = pd.DataFrame(
        selected_fold_metric_rows
    )

    selected_per_class_metrics = pd.concat(
        selected_per_class_frames,
        ignore_index=True,
    )

    selected_per_well_metrics = pd.concat(
        selected_per_well_frames,
        ignore_index=True,
    )

    selected_summary = pd.DataFrame([
        {
            "run_mode": RUN_MODE,
            "outer_folds": int(
                selected_oof_predictions[
                    "outer_fold"
                ].nunique()
            ),
            "rows": int(
                len(
                    selected_oof_predictions
                )
            ),
            "wells": int(
                selected_oof_predictions[
                    "WELL"
                ].nunique()
            ),
            "well_balanced_refined_accuracy": float(
                selected_per_well_metrics[
                    "refined_accuracy"
                ].mean()
            ),
            "well_balanced_refined_balanced_accuracy": float(
                selected_per_well_metrics[
                    "refined_balanced_accuracy"
                ].mean()
            ),
            "well_balanced_refined_macro_f1": float(
                selected_per_well_metrics[
                    "refined_macro_f1_present_classes"
                ].mean()
            ),
            "well_balanced_raw_macro_f1": float(
                selected_per_well_metrics[
                    "raw_macro_f1_present_classes"
                ].mean()
            ),
            "well_balanced_log_loss": float(
                selected_per_well_metrics[
                    "log_loss"
                ].mean()
            ),
            "worst_well_log_loss": float(
                selected_per_well_metrics[
                    "log_loss"
                ].max()
            ),
        }
    ])

    selected_oof_predictions.to_parquet(
        OUTPUT_DIR
        / "06_selected_fused_OOF_predictions.parquet",
        index=False,
    )

    selected_fold_metrics.to_csv(
        OUTPUT_DIR
        / "07_selected_fold_metrics.csv",
        index=False,
    )

    selected_per_class_metrics.to_csv(
        OUTPUT_DIR
        / "08_selected_per_class_metrics.csv",
        index=False,
    )

    selected_per_well_metrics.to_csv(
        OUTPUT_DIR
        / "09_selected_per_well_metrics.csv",
        index=False,
    )

    selected_summary.to_csv(
        OUTPUT_DIR
        / "10_selected_well_balanced_summary.csv",
        index=False,
    )

    display(selected_summary)

Loading cached evidence fold: 1
Loading cached evidence fold: 2
Loading cached evidence fold: 3
Loading cached evidence fold: 4
Loading cached evidence fold: 5


,run_mode,outer_folds,rows,wells,well_balanced_refined_accuracy,well_balanced_refined_balanced_accuracy,well_balanced_refined_macro_f1,well_balanced_raw_macro_f1,well_balanced_log_loss,worst_well_log_loss
0,full_cv,5,1170511,98,0.714595,0.397469,0.38569,0.386646,0.838938,2.00673


In [19]:
if RUN_MODE in {
    "pilot",
    "full_cv",
}:
    expected_rows = int(
        train_raw[
            "GROUP_FOLD"
        ].isin(
            ACTIVE_OUTER_FOLDS
        ).sum()
    )

    final_probability_columns = [
        f"final_prob_{code}"
        for code in CLASS_CODES
    ]

    final_probability_matrix = (
        selected_oof_predictions[
            final_probability_columns
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    stage_gate = {
        "official_test_unused": True,
        "exact_legacy_reproduction_passed": bool(
            legacy_reproduction_gate[
                "overall_reproduction_passed"
            ]
        ),
        "raw_location_classification_or_clustering_not_used": True,
        "X_LOC_removed_from_unrestricted_evidence_RF": True,
        "Y_LOC_removed_from_unrestricted_evidence_RF": True,
        "legacy_causal_feature_engineering_preserved": True,
        "legacy_hybrid_resampling_preserved": True,
        "legacy_RF_hyperparameters_preserved": True,
        "legacy_chalk_refinement_preserved": True,
        "geo_prior_continuous_XY_neighbors": True,
        "geo_prior_uses_vertical_window": True,
        "geo_prior_well_balanced": True,
        "geo_prior_formation_hierarchy": bool(
            USE_FORMATION_HIERARCHY
        ),
        "geo_prior_global_fallback": True,
        "fusion_support_adaptive": True,
        "fusion_distance_adaptive": True,
        "fusion_depth_adaptive": True,
        "fusion_agreement_adaptive": True,
        "fusion_spatial_ratio_bounded": True,
        "outer_well_leakage_zero": bool(
            (
                evidence_fold_metadata[
                    "well_overlap_count"
                ]
                == 0
            ).all()
        ),
        "expected_outer_fold_count": int(
            len(ACTIVE_OUTER_FOLDS)
        ),
        "observed_outer_fold_count": int(
            selected_oof_predictions[
                "outer_fold"
            ].nunique()
        ),
        "outer_fold_count_correct": (
            selected_oof_predictions[
                "outer_fold"
            ].nunique()
            == len(ACTIVE_OUTER_FOLDS)
        ),
        "expected_OOF_rows": expected_rows,
        "observed_OOF_rows": int(
            len(selected_oof_predictions)
        ),
        "OOF_row_count_correct": (
            len(selected_oof_predictions)
            == expected_rows
        ),
        "OOF_row_ids_unique": bool(
            not selected_oof_predictions[
                "ORIGINAL_ROW_ID"
            ].duplicated().any()
        ),
        "final_probabilities_finite": bool(
            np.isfinite(
                final_probability_matrix
            ).all()
        ),
        "final_probabilities_sum_to_one": bool(
            np.allclose(
                final_probability_matrix.sum(
                    axis=1
                ),
                1.0,
                rtol=1e-6,
                atol=1e-6,
            )
        ),
        "run_mode": RUN_MODE,
    }

    stage_gate[
        "overall_geo_stratigraphic_fusion_CV_passed"
    ] = bool(
        all(
            value
            for key, value in (
                stage_gate.items()
            )
            if (
                isinstance(value, bool)
                and key
                != "overall_geo_stratigraphic_fusion_CV_passed"
            )
        )
    )

    print(
        json.dumps(
            stage_gate,
            indent=2,
        )
    )

    if not stage_gate[
        "overall_geo_stratigraphic_fusion_CV_passed"
    ]:
        raise RuntimeError(
            "Geo-stratigraphic fusion CV gate failed."
        )

    with open(
        OUTPUT_DIR
        / "11_CV_stage_gate.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(stage_gate),
            file,
            ensure_ascii=False,
            indent=2,
        )

    frozen_configuration = {
        "stage": (
            "07G_legacy_geo_stratigraphic_fusion"
        ),
        "selected_candidate": {
            key: json_safe(
                selected_candidate[key]
            )
            for key in [
                "candidate_id",
                "k_neighbors",
                "vertical_half_window",
                "lambda_max",
                "spatial_ratio_clip",
                "resampling_prior_correction_power",
                "formation_hierarchy",
            ]
        },
        "selection_rule": (
            "Maximize well-balanced refined macro F1; "
            "then refined balanced accuracy; "
            "then minimize well-balanced log loss; "
            "then prefer smaller lambda."
        ),
        "evidence_features": (
            LEGACY_EVIDENCE_FEATURES
        ),
        "RF_parameters": RF_PARAMETERS,
        "geo_prior_contract": {
            "continuous_XY_neighbors": True,
            "location_clustering": False,
            "vertical_column_rule": (
                "Prefer Z_LOC with sufficient support; "
                "otherwise DEPTH_MD."
            ),
            "formation_hierarchy": bool(
                USE_FORMATION_HIERARCHY
            ),
            "minimum_rows_per_well_window": int(
                MIN_ROWS_PER_WELL_WINDOW
            ),
            "minimum_formation_support_wells": int(
                MIN_FORMATION_SUPPORT_WELLS
            ),
            "minimum_wells_full_trust": int(
                MIN_WELLS_FULL_TRUST
            ),
            "rows_per_well_full_trust": float(
                ROWS_PER_WELL_FULL_TRUST
            ),
            "prior_smoothing_equivalent_wells": float(
                PRIOR_SMOOTHING_EQUIVALENT_WELLS
            ),
        },
        "official_test_used": False,
        "full_five_fold_CV": (
            RUN_MODE == "full_cv"
        ),
    }

    with open(
        OUTPUT_DIR
        / "12_run_configuration.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(
                frozen_configuration
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    if RUN_MODE == "full_cv":
        with open(
            FROZEN_SELECTION_PATH,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                json_safe(
                    frozen_configuration
                ),
                file,
                ensure_ascii=False,
                indent=2,
            )

        print(
            "Frozen full-CV configuration saved:"
        )

        print(FROZEN_SELECTION_PATH)

{
  "official_test_unused": true,
  "exact_legacy_reproduction_passed": true,
  "raw_location_classification_or_clustering_not_used": true,
  "X_LOC_removed_from_unrestricted_evidence_RF": true,
  "Y_LOC_removed_from_unrestricted_evidence_RF": true,
  "legacy_causal_feature_engineering_preserved": true,
  "legacy_hybrid_resampling_preserved": true,
  "legacy_RF_hyperparameters_preserved": true,
  "legacy_chalk_refinement_preserved": true,
  "geo_prior_continuous_XY_neighbors": true,
  "geo_prior_uses_vertical_window": true,
  "geo_prior_well_balanced": true,
  "geo_prior_formation_hierarchy": true,
  "geo_prior_global_fallback": true,
  "fusion_support_adaptive": true,
  "fusion_distance_adaptive": true,
  "fusion_depth_adaptive": true,
  "fusion_agreement_adaptive": true,
  "fusion_spatial_ratio_bounded": true,
  "outer_well_leakage_zero": true,
  "expected_outer_fold_count": 5,
  "observed_outer_fold_count": 5,
  "outer_fold_count_correct": true,
  "expected_OOF_rows": 1170511,
  "ob

In [20]:
if (
    RUN_MODE
    in {
        "pilot",
        "full_cv",
    }
    and LEGACY_OOF_SUMMARY_PATH.exists()
):
    legacy_oof_summary = pd.read_csv(
        LEGACY_OOF_SUMMARY_PATH
    )

    print(
        "02B exact Legacy grouped OOF:"
    )

    display(legacy_oof_summary)

    print(
        "07G selected fusion:"
    )

    display(selected_summary)

02B exact Legacy grouped OOF:


,prediction_variant,accuracy,balanced_accuracy,macro_f1_present_classes,macro_f1_all_12_classes,weighted_f1,macro_precision_present_classes,macro_recall_present_classes,log_loss
0,raw,0.724459,0.310379,0.342907,0.342907,0.685022,0.478220,0.310379,0.830011
1,legacy_refined,0.723001,0.339798,0.367302,0.367302,0.685164,0.444439,0.339798,NaN


07G selected fusion:


,run_mode,outer_folds,rows,wells,well_balanced_refined_accuracy,well_balanced_refined_balanced_accuracy,well_balanced_refined_macro_f1,well_balanced_raw_macro_f1,well_balanced_log_loss,worst_well_log_loss
0,full_cv,5,1170511,98,0.714595,0.397469,0.38569,0.386646,0.838938,2.00673


In [21]:
if RUN_MODE == "final_test":
    with open(
        FROZEN_SELECTION_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        frozen_configuration = (
            json.load(file)
        )

    if not frozen_configuration.get(
        "full_five_fold_CV",
        False,
    ):
        raise RuntimeError(
            "Frozen configuration was not created by full five-fold CV."
        )

    selected = frozen_configuration[
        "selected_candidate"
    ]

    print("Frozen candidate:")
    print(
        json.dumps(
            selected,
            indent=2,
        )
    )

    # test.csv is read only after the full-CV configuration is frozen.
    test_raw = load_csv_full(
        TEST_PATH
    )

    test_raw[
        "ORIGINAL_ROW_ID"
    ] = np.arange(
        len(test_raw),
        dtype=np.int64,
    )

    if TARGET_COLUMN not in (
        test_raw.columns
    ):
        raise RuntimeError(
            "test.csv has no labels, so metrics cannot be computed."
        )

    raw_full_train = (
        train_raw.drop(
            columns=[
                "GROUP_FOLD"
            ]
        )
        .copy()
        .reset_index(drop=True)
    )

    raw_test = (
        test_raw.copy()
        .reset_index(drop=True)
    )

    train_wells = set(
        raw_full_train[
            "WELL"
        ].dropna()
    )

    test_wells = set(
        raw_test[
            "WELL"
        ].dropna()
    )

    overlap = train_wells & test_wells

    if overlap:
        raise RuntimeError(
            "Train/test WELL overlap detected: "
            + str(
                sorted(overlap)
            )
        )

    prepared = (
        prepare_evidence_train_and_evaluation(
            raw_full_train,
            raw_test,
        )
    )

    fitted = fit_evidence_RF(
        prepared
    )

    bank = build_geo_prior_bank(
        raw_full_train
    )

    geo_result = compute_geo_prior(
        raw_test,
        bank,
        k_neighbors=int(
            selected[
                "k_neighbors"
            ]
        ),
        vertical_half_window=float(
            selected[
                "vertical_half_window"
            ]
        ),
    )

    fusion = (
        fuse_evidence_and_geo_prior(
            evidence_probabilities=(
                fitted[
                    "probabilities"
                ]
            ),
            raw_training_prior=(
                prepared[
                    "raw_training_prior"
                ]
            ),
            resampled_training_prior=(
                prepared[
                    "resampled_training_prior"
                ]
            ),
            geo_prior=(
                geo_result[
                    "prior"
                ]
            ),
            geo_global_prior=(
                bank[
                    "global_well_balanced_prior"
                ]
            ),
            reliability_base=(
                geo_result[
                    "reliability_base"
                ]
            ),
            lambda_max=float(
                selected[
                    "lambda_max"
                ]
            ),
            spatial_ratio_clip=float(
                selected[
                    "spatial_ratio_clip"
                ]
            ),
            resampling_prior_correction_power=float(
                selected[
                    "resampling_prior_correction_power"
                ]
            ),
        )
    )

    final_probabilities = (
        fusion[
            "final_probabilities"
        ]
    )

    (
        raw_predictions,
        refined_predictions,
        refinement_mask,
    ) = apply_legacy_chalk_refinement(
        final_probabilities
    )

    y_test = prepared[
        "y_evaluation"
    ]

    raw_test_metrics = (
        calculate_metrics(
            y_test,
            raw_predictions,
            probabilities=(
                final_probabilities
            ),
        )
    )

    refined_test_metrics = (
        calculate_metrics(
            y_test,
            refined_predictions,
        )
    )

    final_test_summary = pd.DataFrame([
        {
            "prediction_variant": (
                "07G_GEO_FUSION_RAW"
            ),
            **raw_test_metrics,
        },
        {
            "prediction_variant": (
                "07G_GEO_FUSION_LEGACY_REFINED"
            ),
            **refined_test_metrics,
        },
    ])

    final_test_per_class_raw = (
        calculate_per_class_metrics(
            y_test,
            raw_predictions,
        )
    )

    final_test_per_class_raw[
        "prediction_variant"
    ] = "07G_GEO_FUSION_RAW"

    final_test_per_class_refined = (
        calculate_per_class_metrics(
            y_test,
            refined_predictions,
        )
    )

    final_test_per_class_refined[
        "prediction_variant"
    ] = (
        "07G_GEO_FUSION_LEGACY_REFINED"
    )

    final_test_per_class = pd.concat(
        [
            final_test_per_class_raw,
            final_test_per_class_refined,
        ],
        ignore_index=True,
    )

    test_metadata = (
        prepared[
            "evaluation_metadata"
        ]
        .copy()
    )

    final_test_per_well = (
        calculate_per_well_metrics(
            test_metadata,
            final_probabilities,
            raw_predictions,
            refined_predictions,
        )
    )

    final_test_predictions = (
        test_metadata.copy()
    )

    final_test_predictions[
        "raw_fused_prediction_index"
    ] = raw_predictions

    final_test_predictions[
        "legacy_refined_fused_prediction_index"
    ] = refined_predictions

    final_test_predictions[
        "legacy_refinement_modified"
    ] = refinement_mask

    final_test_predictions[
        "adaptive_lambda"
    ] = fusion[
        "row_lambda"
    ]

    final_test_predictions[
        "geo_nearest_distance"
    ] = geo_result[
        "nearest_distance"
    ]

    final_test_predictions[
        "geo_support_wells"
    ] = geo_result[
        "support_wells"
    ]

    final_test_predictions[
        "geo_average_rows_per_well"
    ] = geo_result[
        "average_rows"
    ]

    final_test_predictions[
        "geo_agreement"
    ] = geo_result[
        "agreement"
    ]

    final_test_predictions[
        "geo_formation_specific"
    ] = geo_result[
        "formation_specific"
    ]

    for class_index, class_code in enumerate(
        CLASS_CODES
    ):
        final_test_predictions[
            f"final_prob_{class_code}"
        ] = (
            final_probabilities[
                :,
                class_index,
            ].astype(
                np.float32
            )
        )

        final_test_predictions[
            f"geo_prior_{class_code}"
        ] = (
            geo_result[
                "prior"
            ][
                :,
                class_index,
            ].astype(
                np.float32
            )
        )

    final_test_summary.to_csv(
        OUTPUT_DIR
        / "01_final_test_summary.csv",
        index=False,
    )

    final_test_per_class.to_csv(
        OUTPUT_DIR
        / "02_final_test_per_class.csv",
        index=False,
    )

    final_test_per_well.to_csv(
        OUTPUT_DIR
        / "03_final_test_per_well.csv",
        index=False,
    )

    final_test_predictions.to_parquet(
        OUTPUT_DIR
        / "04_final_test_predictions.parquet",
        index=False,
    )

    prepared[
        "resampling_audit"
    ].to_csv(
        OUTPUT_DIR
        / "05_final_train_resampling_audit.csv",
        index=False,
    )

In [22]:
if RUN_MODE == "final_test":
    distance_table = (
        final_test_predictions[
            [
                "ORIGINAL_ROW_ID",
                "WELL",
                "LITHOLOGY_ENCODED",
                "legacy_refined_fused_prediction_index",
                "adaptive_lambda",
                "geo_nearest_distance",
                "geo_support_wells",
                "geo_average_rows_per_well",
                "geo_agreement",
            ]
        ]
        .copy()
    )

    finite_distance = distance_table[
        "geo_nearest_distance"
    ].notna()

    if finite_distance.any():
        distance_table.loc[
            finite_distance,
            "distance_quartile",
        ] = pd.qcut(
            distance_table.loc[
                finite_distance,
                "geo_nearest_distance",
            ],
            q=4,
            duplicates="drop",
        ).astype(str)

        distance_rows = []

        for distance_group, group_frame in (
            distance_table.loc[
                finite_distance
            ]
            .groupby(
                "distance_quartile",
                observed=True,
            )
        ):
            y_group = group_frame[
                "LITHOLOGY_ENCODED"
            ].to_numpy(
                dtype=int
            )

            prediction_group = (
                group_frame[
                    "legacy_refined_fused_prediction_index"
                ]
                .to_numpy(
                    dtype=int
                )
            )

            present_labels = sorted(
                np.unique(
                    y_group
                ).tolist()
            )

            distance_rows.append({
                "distance_quartile": (
                    distance_group
                ),
                "rows": int(
                    len(group_frame)
                ),
                "wells": int(
                    group_frame[
                        "WELL"
                    ].nunique()
                ),
                "mean_nearest_distance": float(
                    group_frame[
                        "geo_nearest_distance"
                    ].mean()
                ),
                "mean_lambda": float(
                    group_frame[
                        "adaptive_lambda"
                    ].mean()
                ),
                "refined_accuracy": float(
                    accuracy_score(
                        y_group,
                        prediction_group,
                    )
                ),
                "refined_balanced_accuracy": float(
                    balanced_accuracy_score(
                        y_group,
                        prediction_group,
                    )
                ),
                "refined_macro_f1_present_classes": float(
                    f1_score(
                        y_group,
                        prediction_group,
                        labels=present_labels,
                        average="macro",
                        zero_division=0,
                    )
                ),
            })

        distance_stratified_metrics = (
            pd.DataFrame(
                distance_rows
            )
        )

        distance_stratified_metrics.to_csv(
            OUTPUT_DIR
            / "06_test_distance_stratified_metrics.csv",
            index=False,
        )

    comparison_frames = []

    if LEGACY_TEST_SUMMARY_PATH.exists():
        legacy_test_summary = pd.read_csv(
            LEGACY_TEST_SUMMARY_PATH
        )

        legacy_test_summary[
            "model_source"
        ] = "02A_exact_legacy"

        comparison_frames.append(
            legacy_test_summary
        )

    fusion_test_comparison = (
        final_test_summary.copy()
    )

    fusion_test_comparison[
        "model_source"
    ] = (
        "07G_geo_stratigraphic_fusion"
    )

    comparison_frames.append(
        fusion_test_comparison
    )

    official_test_comparison = pd.concat(
        comparison_frames,
        ignore_index=True,
        sort=False,
    )

    official_test_comparison.to_csv(
        OUTPUT_DIR
        / "07_official_test_comparison_with_02A.csv",
        index=False,
    )

    display(
        official_test_comparison
    )

    final_probability_matrix = (
        final_test_predictions[
            [
                f"final_prob_{code}"
                for code in CLASS_CODES
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    final_test_gate = {
        "official_test_used": True,
        "official_test_used_after_full_CV_freeze": True,
        "frozen_candidate_unchanged": True,
        "train_test_WELL_overlap_zero": (
            len(overlap) == 0
        ),
        "legacy_causal_features_preserved": True,
        "legacy_hybrid_resampling_preserved": True,
        "geo_prior_training_labels_only": True,
        "geo_prior_continuous_XY_plus_vertical_context": True,
        "location_clustering_not_used": True,
        "test_rows": int(
            len(
                final_test_predictions
            )
        ),
        "test_wells": int(
            final_test_predictions[
                "WELL"
            ].nunique()
        ),
        "probabilities_finite": bool(
            np.isfinite(
                final_probability_matrix
            ).all()
        ),
        "probabilities_sum_to_one": bool(
            np.allclose(
                final_probability_matrix.sum(
                    axis=1
                ),
                1.0,
                rtol=1e-6,
                atol=1e-6,
            )
        ),
    }

    final_test_gate[
        "overall_final_test_stage_passed"
    ] = bool(
        all(
            value
            for key, value in (
                final_test_gate.items()
            )
            if (
                isinstance(value, bool)
                and key
                != "overall_final_test_stage_passed"
            )
        )
    )

    if not final_test_gate[
        "overall_final_test_stage_passed"
    ]:
        raise RuntimeError(
            "Final test gate failed."
        )

    with open(
        OUTPUT_DIR
        / "08_final_test_gate.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(
                final_test_gate
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    final_test_run_configuration = {
        "frozen_configuration": (
            frozen_configuration
        ),
        "train_rows_raw": int(
            prepared[
                "raw_training_rows"
            ]
        ),
        "train_rows_resampled": int(
            prepared[
                "resampled_training_rows"
            ]
        ),
        "test_rows": int(
            len(
                final_test_predictions
            )
        ),
        "evidence_fit_seconds": float(
            fitted[
                "fit_seconds"
            ]
        ),
        "vertical_column": (
            bank[
                "vertical_column"
            ]
        ),
        "official_test_used": True,
    }

    with open(
        OUTPUT_DIR
        / "09_final_test_run_configuration.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(
                final_test_run_configuration
            ),
            file,
            ensure_ascii=False,
            indent=2,
        )

    sentinel = {
        "official_test_consumed": True,
        "frozen_candidate_id": (
            selected[
                "candidate_id"
            ]
        ),
        "instruction": (
            "Do not alter the model after inspecting "
            "the official-test result."
        ),
    }

    with open(
        FINAL_TEST_SENTINEL_PATH,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            sentinel,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print(
        "\nFinal test completed and locked."
    )

In [23]:
class_mapping.to_csv(
    OUTPUT_DIR
    / "00_class_mapping.csv",
    index=False,
)

feature_registry.to_csv(
    OUTPUT_DIR
    / "00_legacy_to_evidence_feature_registry.csv",
    index=False,
)

candidate_registry.to_csv(
    OUTPUT_DIR
    / "00_fusion_candidate_registry.csv",
    index=False,
)

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
}

with open(
    OUTPUT_DIR
    / "99_environment.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Outputs saved in:")
print(OUTPUT_DIR)

Outputs saved in:
<LOCAL_PATH_REDACTED>
